# Script 4 — Avaliação Aprofundada, Análise de Risco e Estresse Probabilístico

**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Este script consome os artefatos do `03_cvm_modelagem.ipynb` e realiza avaliação
completa sobre **todos os 36 targets treinados** (9 variáveis × 4 horizontes),
além das análises de risco corporativo.

| Etapa | Conteúdo |
|-------|----------|
| 0 | Imports, logging, constantes espelhadas do Script 3 |
| 1 | Carga de todos os artefatos do Script 3 |
| 2 | Funções de métricas com inversão de transformações (escala original R$ mil) |
| 3 | Relatório consolidado — todos os 36 targets × 4 algoritmos |
| 4 | Diagnóstico CV vs. Teste (detecção de overfitting por target) |
| 5 | Análise estratificada por setor — todos os 36 targets |
| 6 | Z''-Score de Altman histórico e prospectivo |
| 7 | Score de Risco Composto (0–100, 8 gatilhos) |
| 8 | Análise de Estresse Monte Carlo (500 sim., σ=15%) |
| 8b | Conformal Prediction — IC 90% com garantia estatística |
| 9 | Feature Importance agregada com ranking por família |
| 10 | Análise de resíduos + teste de viés sistemático |
| 11 | Visualizações (6 figuras) |
| 12 | Persistência de todos os artefatos para o Script 5 |

---
**Referências:**
- Z''-Score: Altman (1968, 1995) | SMAPE: Hyndman & Athanasopoulos (2018)
- RobustScaler / inversão de transformações: Pedregosa et al. (2011)
- Análise de estresse σ=15%: Damodaran (2012)

## Etapa 0. Imports, Configuração e Logging

In [1]:
import json
import logging
import pickle
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as sp_stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 180)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
(PASTA_SAIDA / 'logs').mkdir(exist_ok=True)
(PASTA_SAIDA / 'figuras').mkdir(exist_ok=True)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
STAGE_LOGS = []

# ── Logging ───────────────────────────────────────────────────────────────────
logger = logging.getLogger('pipeline_avaliacao_v4')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'pipeline_avaliacao.log',
                           mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

def registrar_evento(etapa, mensagem, nivel='info', **dados):
    registro = {
        'run_id': RUN_ID,
        'etapa': etapa,
        'mensagem': mensagem,
        'nivel': nivel,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }
    if dados:
        registro.update(dados)
    STAGE_LOGS.append(registro)
    log_fn = getattr(logger, nivel, logger.info)
    if dados:
        log_fn('[%s] %s | %s', etapa, mensagem, dados)
    else:
        log_fn('[%s] %s', etapa, mensagem)
    return registro

def resumo_df(nome, df, colunas_criticas=None):
    colunas_criticas = colunas_criticas or []
    if df is None:
        registrar_evento(nome, 'dataframe ausente', nivel='warning')
        return
    faltantes = [c for c in colunas_criticas if c not in df.columns]
    resumo = {
        'linhas': int(len(df)),
        'colunas': int(df.shape[1]),
        'colunas_criticas_ausentes': faltantes,
        'nulos_total': int(df.isna().sum().sum()) if not df.empty else 0,
    }
    registrar_evento(nome, 'resumo do dataframe', **resumo)
    print(f'[{nome}] linhas={resumo["linhas"]:,} | colunas={resumo["colunas"]} | nulos={resumo["nulos_total"]:,}')
    if faltantes:
        print(f'[{nome}] colunas críticas ausentes: {faltantes}')

# ── Constantes IDÊNTICAS ao Script 3 ─────────────────────────────────────────
# Devem ser mantidas em sincronia com 03_cvm_modelagem.ipynb
_TARGET_BASES = [
    'DRE_3.01', 'DRE_3.11', 'EBITDA',
    'BPA_1', 'BPA_1.01', 'BPP_2.01', 'BPP_2.03', 'BPP_2',
    'DFC_MI_6.01',
]
_HORIZONTES = ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']

# 9 variáveis × 4 horizontes = 36 targets treinados
TODOS_TARGETS = [f'TARGET_{b}{h}' for b in _TARGET_BASES for h in _HORIZONTES]

# Transformações por base (log1p → targets positivos | arcsinh → podem ser negativos)
_LOG_BASES     = {'DRE_3.01','EBITDA','BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2'}
_ARCSINH_BASES = {'DFC_MI_6.01','DRE_3.11'}
LOG_TARGETS     = {f'TARGET_{b}{h}' for b in _LOG_BASES     for h in _HORIZONTES}
ARCSINH_TARGETS = {f'TARGET_{b}{h}' for b in _ARCSINH_BASES for h in _HORIZONTES}

# Nomes legíveis para relatórios
NOME_VARIAVEL = {
    'DRE_3.01':    'Receita Líquida',
    'DRE_3.11':    'Lucro Líquido',
    'EBITDA':      'EBITDA',
    'BPA_1':       'Ativo Total',
    'BPA_1.01':    'Ativo Circulante',
    'BPP_2.01':    'Passivo Circulante',
    'BPP_2.03':    'Patrimônio Líquido',
    'BPP_2':       'Passivo Total',
    'DFC_MI_6.01': 'FCO',
}

# Foco primário do TCC (Receita, Lucro, EBITDA) — usados em análises detalhadas
BASES_FOCO  = ['DRE_3.01','DRE_3.11','EBITDA']
TARGETS_FOCO = [f'TARGET_{b}{h}' for b in BASES_FOCO for h in _HORIZONTES]  # 12 targets

# Parâmetros de risco e estresse
ZONA_SEGURA    = 2.60
ZONA_CINZA_INF = 1.10
N_SIM_MC       = 500
SIGMA_MC       = 0.15
SEED_MC        = 42
COVID_ANOS     = {2020, 2021}

registrar_evento(
    'Etapa 0',
    'Script 4 inicializado',
    targets_totais=len(TODOS_TARGETS),
    targets_foco=len(TARGETS_FOCO),
    run_id=RUN_ID,
)
print(f'✅ Etapa 0 — {len(TODOS_TARGETS)} targets configurados ({len(TARGETS_FOCO)} foco TCC) | RUN_ID={RUN_ID}')


2026-06-05 16:34:59 | INFO     | [Etapa 0] Script 4 inicializado | {'targets_totais': 36, 'targets_foco': 12, 'run_id': '20260605_163459'}


✅ Etapa 0 — 36 targets configurados (12 foco TCC) | RUN_ID=20260605_163459


## Etapa 1. Carga dos Artefatos do Script 3

In [2]:
def _pkl(nome, req=True):
    p = PASTA_SAIDA / nome
    if not p.exists():
        msg = f'{nome} não encontrado — execute 03_cvm_modelagem.ipynb antes.'
        if req:
            raise FileNotFoundError(msg)
        registrar_evento('artefatos', msg, nivel='warning', arquivo=nome)
        return None
    with open(p, 'rb') as f:
        obj = pickle.load(f)
    registrar_evento('artefatos', f'artefato carregado: {nome}', arquivo=nome)
    return obj

def _load_csv(nome, req=False):
    p = PASTA_SAIDA / nome
    if not p.exists():
        msg = f'{nome} não encontrado'
        if req:
            raise FileNotFoundError(msg)
        registrar_evento('artefatos', msg, nivel='warning', arquivo=nome)
        return pd.DataFrame()
    df = pd.read_csv(p)
    registrar_evento('artefatos', f'CSV carregado: {nome}', arquivo=nome, linhas=int(len(df)), colunas=int(df.shape[1]))
    if df.empty:
        logger.warning('%s está vazio', nome)
    return df

def _load_parquet(nome, req=False):
    p = PASTA_SAIDA / nome
    if not p.exists():
        msg = f'{nome} não encontrado'
        if req:
            raise FileNotFoundError(msg)
        registrar_evento('artefatos', msg, nivel='warning', arquivo=nome)
        return pd.DataFrame()
    df = pd.read_parquet(p)
    registrar_evento('artefatos', f'Parquet carregado: {nome}', arquivo=nome, linhas=int(len(df)), colunas=int(df.shape[1]))
    if df.empty:
        logger.warning('%s está vazio', nome)
    return df

# Artefatos obrigatórios
treino  = _load_parquet('treino.parquet', req=True)
teste   = _load_parquet('teste.parquet', req=True)
melhores                     = _pkl('melhores_modelos.pkl')
metricas_teste_pkl           = _pkl('metricas_teste.pkl')
baselines                    = _pkl('baselines.pkl')
feature_importances          = _pkl('feature_importances.pkl')
selected_features_por_target = _pkl('selected_features_por_target.pkl')

# Opcionais (enriquecem análises quando disponíveis)
FEATURES          = _pkl('features.pkl', req=False) or []
KPIS              = _pkl('kpis.pkl', req=False) or []
TARGETS_POR_HOR   = _pkl('targets_por_horizonte.pkl', req=False) or {}
TARGET_COLS_SOURCE= _pkl('target_cols_source.pkl', req=False) or {}

# CSVs de resultado do Script 3
df_cv = _load_csv('resultados_cv.csv', req=False)
df_te = _load_csv('resultados_teste.csv', req=False)

# Predições detalhadas linha a linha (geradas na Etapa 6 do Script 3)
df_pred = _load_parquet('predicoes_teste_detalhadas.parquet', req=False)

# Predições prospectivas 2026 (geradas na Etapa Prospectiva do Script 3)
df_prosp = _load_parquet('predicoes_prospectivas.parquet', req=False)

# TARGETS ativos: apenas os que foram de fato treinados (presentes em melhores)
TARGETS = [t for t in TODOS_TARGETS if t in melhores]

# Dataset consolidado para Z-Score (usa parquet se disponível, senão concatena splits)
dataset = _load_parquet('dataset_cvm_consolidado.parquet', req=False)
if dataset.empty:
    dataset = pd.concat([treino, teste], ignore_index=True)
    registrar_evento('artefatos', 'dataset consolidado reconstruído por concatenação treino+teste', linhas=int(len(dataset)))

# Normaliza datas (remove timezone para evitar erros em joins)
for _df in [treino, teste, dataset]:
    for _c in ('DT_REFER', 'DT_TARGET', 'DT_TARGET_DFP'):
        if _c in _df.columns:
            _df[_c] = pd.to_datetime(_df[_c], utc=True, errors='coerce').dt.tz_localize(None)

# Mapas de lookup CNPJ → metadados
mapa_setor = {}
mapa_nome = {}
if 'CNPJ_CIA' in dataset.columns:
    _b = dataset.drop_duplicates('CNPJ_CIA')
    if 'SETOR' in _b.columns:
        mapa_setor = _b.set_index('CNPJ_CIA')['SETOR'].to_dict()
    if 'NOME_CIA' in _b.columns:
        mapa_nome = _b.set_index('CNPJ_CIA')['NOME_CIA'].to_dict()

# Resumos de base
resumo_df('treino', treino, colunas_criticas=['CNPJ_CIA', 'ANO'])
resumo_df('teste', teste, colunas_criticas=['CNPJ_CIA', 'ANO'])
resumo_df('dataset', dataset, colunas_criticas=['CNPJ_CIA', 'ANO', 'SETOR'])
resumo_df('df_pred', df_pred, colunas_criticas=['CNPJ_CIA', 'Target', 'Algoritmo', 'y_true', 'y_pred'])

print(f'Targets treinados       : {len(TARGETS)} / {len(TODOS_TARGETS)} possíveis')
print(f'Predições detalhadas    : {len(df_pred):,} linhas')
print(f'Predições prospectivas  : {len(df_prosp):,} linhas')
print(f'Algoritmos vencedores   : {sorted(set(melhores.values())) if melhores else []}')
print(f'Setores identificados   : {sorted(set(mapa_setor.values())) if mapa_setor else []}')

registrar_evento(
    'artefatos',
    'artefatos principais carregados',
    treino=int(len(treino)),
    teste=int(len(teste)),
    dataset=int(len(dataset)),
    predicoes=int(len(df_pred)),
    prospectivas=int(len(df_prosp)),
    targets_treinados=int(len(TARGETS)),
)


2026-06-05 16:35:03 | INFO     | [artefatos] Parquet carregado: treino.parquet | {'arquivo': 'treino.parquet', 'linhas': 813, 'colunas': 991}
2026-06-05 16:35:03 | INFO     | [artefatos] Parquet carregado: teste.parquet | {'arquivo': 'teste.parquet', 'linhas': 193, 'colunas': 991}
2026-06-05 16:35:03 | INFO     | [artefatos] artefato carregado: melhores_modelos.pkl | {'arquivo': 'melhores_modelos.pkl'}
2026-06-05 16:35:03 | INFO     | [artefatos] artefato carregado: metricas_teste.pkl | {'arquivo': 'metricas_teste.pkl'}
2026-06-05 16:35:03 | INFO     | [artefatos] artefato carregado: baselines.pkl | {'arquivo': 'baselines.pkl'}
2026-06-05 16:35:03 | INFO     | [artefatos] artefato carregado: feature_importances.pkl | {'arquivo': 'feature_importances.pkl'}
2026-06-05 16:35:03 | INFO     | [artefatos] artefato carregado: selected_features_por_target.pkl | {'arquivo': 'selected_features_por_target.pkl'}
2026-06-05 16:35:03 | INFO     | [artefatos] artefato carregado: features.pkl | {'arqu

[treino] linhas=813 | colunas=991 | nulos=83,783
[teste] linhas=193 | colunas=991 | nulos=17,669
[dataset] linhas=1,031 | colunas=775 | nulos=392,616
[df_pred] linhas=28,035 | colunas=7 | nulos=0
Targets treinados       : 36 / 36 possíveis
Predições detalhadas    : 28,035 linhas
Predições prospectivas  : 828 linhas
Algoritmos vencedores   : ['GradientBoosting', 'RandomForest']
Setores identificados   : ['Commodities', 'Energia', 'Petróleo', 'Tecnologia', 'Varejo']


{'run_id': '20260605_163459',
 'etapa': 'artefatos',
 'mensagem': 'artefatos principais carregados',
 'nivel': 'info',
 'timestamp': '2026-06-05 16:35:03',
 'treino': 813,
 'teste': 193,
 'dataset': 1031,
 'predicoes': 28035,
 'prospectivas': 828,
 'targets_treinados': 36}

## Etapa 2. Funções de Métricas com Inversão de Transformações

Todas as métricas são calculadas na **escala original em R$ mil**, após inversão
das transformações aplicadas no Script 3 (`expm1` para log1p; `sinh` para arcsinh).
Isso garante interpretabilidade financeira direta (Hyndman & Athanasopoulos, 2018).

In [3]:
def get_transform(target):
    if target in LOG_TARGETS:
        return 'log1p'
    if target in ARCSINH_TARGETS:
        return 'arcsinh'
    return 'none'

def inv_transform(y, t='none'):
    y = np.asarray(y, float)
    if t == 'log1p':
        return np.expm1(y)
    if t == 'arcsinh':
        return np.sinh(y)
    return y

def alinhar_series(target, yt, yp, modo='auto'):
    """Alinha y_true e y_pred para a mesma escala e devolve metadados de escolha."""
    yt = np.asarray(yt, float)
    yp = np.asarray(yp, float)
    t = get_transform(target)

    def _score(y_true_alinhado, y_pred_alinhado):
        mask = np.isfinite(y_true_alinhado) & np.isfinite(y_pred_alinhado)
        n = int(mask.sum())
        if n < 2:
            return {'n': n, 'score': -1e9, 'ratio': np.nan, 'mask': mask}
        yy = y_true_alinhado[mask]
        pp = y_pred_alinhado[mask]
        med_y = np.nanmedian(np.abs(yy))
        med_p = np.nanmedian(np.abs(pp))
        ratio = med_p / (med_y + 1e-9) if np.isfinite(med_y) else np.nan
        score = float(n)
        if np.isfinite(ratio):
            if 0.25 <= ratio <= 4.0:
                score += 3.0
            elif 0.10 <= ratio <= 10.0:
                score += 1.0
            else:
                score -= 2.0
        if np.nanstd(yy) > 0 and np.nanstd(pp) > 0:
            score += 1.0
        return {'n': n, 'score': score, 'ratio': ratio, 'mask': mask}

    candidatos = []
    if modo in ('auto', 'ytrue_inv_rawpred'):
        yta = inv_transform(yt, t)
        ypa = yp.copy()
        s = _score(yta, ypa)
        candidatos.append({
            'modo': 'ytrue_inv_rawpred',
            'yt': yta[s['mask']],
            'yp': ypa[s['mask']],
            'n': s['n'],
            'score': s['score'],
            'ratio': s['ratio'],
            'transform': t,
        })

    if modo in ('auto', 'raw_raw'):
        ytb = yt.copy()
        ypb = yp.copy()
        s = _score(ytb, ypb)
        candidatos.append({
            'modo': 'raw_raw',
            'yt': ytb[s['mask']],
            'yp': ypb[s['mask']],
            'n': s['n'],
            'score': s['score'],
            'ratio': s['ratio'],
            'transform': t,
        })

    candidatos = [c for c in candidatos if c['n'] >= 2]
    if not candidatos:
        return {
            'modo': 'sem_dados',
            'yt': np.array([]),
            'yp': np.array([]),
            'n': 0,
            'score': -1e9,
            'ratio': np.nan,
            'transform': t,
            'candidatos': [],
        }

    candidatos = sorted(candidatos, key=lambda d: (d['score'], d['n']), reverse=True)
    melhor = candidatos[0]
    melhor['candidatos'] = candidatos
    return melhor

def _smape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    d = (np.abs(yt)+np.abs(yp))/2.0
    m = d > 1e-9
    return float(np.mean(np.abs(yt[m]-yp[m])/d[m])) if m.sum()>0 else np.nan

def _mape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    m = np.abs(yt) > 1e-9
    return float(np.mean(np.abs((yt[m]-yp[m])/yt[m]))) if m.sum()>0 else np.nan

def _rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt,float)-np.asarray(yp,float))**2)))

def _mae(yt, yp):
    return float(np.mean(np.abs(np.asarray(yt,float)-np.asarray(yp,float))))

def _r2(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2 or np.isclose(np.var(yt),0):
        return np.nan
    ss_r = np.sum((yt-yp)**2)
    ss_t = np.sum((yt-yt.mean())**2)
    return float(1-ss_r/ss_t) if ss_t>0 else np.nan

def _theil(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2:
        return np.nan
    em = np.sqrt(np.mean((yt[1:]-yp[1:])**2))
    en = np.sqrt(np.mean((yt[1:]-yt[:-1])**2))
    return float(em/en) if en>1e-9 else np.nan

def _da(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2:
        return np.nan
    return float(np.mean(np.sign(yt[1:]-yt[:-1])==np.sign(yp[1:]-yp[:-1])))

def metricas(yt, yp, target=None):
    """Retorna métricas na escala já alinhada. Se target for dado, aplica a transformação esperada."""
    if target:
        t = get_transform(target)
        yt = inv_transform(yt, t)
        yp = inv_transform(yp, t)
    return {
        'SMAPE': _smape(yt, yp),
        'MAPE': _mape(yt, yp),
        'RMSE': _rmse(yt, yp),
        'MAE': _mae(yt, yp),
        'R2': _r2(yt, yp),
        'TheilU': _theil(yt, yp),
        'DA': _da(yt, yp),
        'N': int(len(yt)),
    }

def metricas_macro(df, target, yt_col='y_true', yp_col='y_pred', grp='CNPJ_CIA'):
    """Média aritmética das métricas por empresa, com ordenação temporal quando disponível."""
    rows = []
    ord_cols = [c for c in ['ANO', 'DT_REFER', 'DT_TARGET', 'DT_TARGET_DFP'] if c in df.columns]

    for _, g in df.groupby(grp):
        if ord_cols:
            g = g.sort_values(ord_cols)
        yt = g[yt_col].values
        yp = g[yp_col].values
        mask = np.isfinite(yt) & np.isfinite(yp)
        if mask.sum() < 2:
            continue
        rows.append(metricas(yt[mask], yp[mask], target))

    if not rows:
        return {}

    df_m = pd.DataFrame(rows)
    return {f'{k}_macro': float(df_m[k].mean()) for k in df_m.columns if k != 'N'}

print('✅ Etapa 2 — funções de métricas e alinhamento prontos')


✅ Etapa 2 — funções de métricas e alinhamento prontos


## Etapa 3. Relatório Consolidado — Todos os 36 Targets × 4 Algoritmos

Consolida as métricas já calculadas pelo Script 3 e adiciona colunas derivadas:
cobertura de baseline, ranking de algoritmo por target e sinalização de overfitting.

In [4]:
print('='*100)
print(f'DESEMPENHO NO HOLD-OUT 2024–2025 — TODOS OS {len(TARGETS)} TARGETS TREINADOS')
print('='*100)

_mc_cv  = 'SMAPE_CV_macro_empresa'
_mc_te  = 'SMAPE_teste_macro_empresa'
_mc_theil = 'TheilU_teste_macro_empresa'
_mc_da  = 'DA_teste_macro_empresa'
_mc_r2  = 'R2_teste_macro_empresa'

if not df_te.empty:
    registrar_evento('Etapa 3', 'iniciando análise de hold-out', linhas=int(len(df_te)), colunas=int(df_te.shape[1]))
    # ── 3.1 Tabela completa por horizonte × algoritmo ─────────────────────────
    cols_disp = [c for c in [_mc_te,'RMSE_teste_macro_empresa',_mc_r2,_mc_theil,_mc_da]
                 if c in df_te.columns]

    for horizonte in _HORIZONTES:
        df_h = df_te[df_te['Horizonte']==horizonte] if 'Horizonte' in df_te.columns else df_te
        if df_h.empty: continue
        print(f'\n--- Horizonte: {horizonte.replace("_","")} ---')
        if cols_disp:
            g = df_h.groupby('Algoritmo')[cols_disp].mean()
            print(g.round(4).to_string())

    # ── 3.2 Contagem de vitórias (todos os 36 targets) ────────────────────────
    print('\n' + '='*70)
    print(f'RANKING DE ALGORITMOS — {len(melhores)} targets (critério: SMAPE_CV macro)')
    print('='*70)
    cnt = Counter(melhores.values())
    for alg, n in sorted(cnt.items(), key=lambda x:-x[1]):
        pct = n/len(melhores)*100
        print(f'  {alg:<22} {n:>3}× ({pct:5.1f}%) {"█"*int(pct/3)}')

    # ── 3.3 Contagem por grupo de variável ────────────────────────────────────
    print('\n--- Vitórias por grupo de variável ---')
    for base in _TARGET_BASES:
        vits = {h: melhores.get(f'TARGET_{base}{h}','?') for h in _HORIZONTES}
        linha = '  '.join(f'{h.replace("_","")}:{v}' for h,v in vits.items())
        print(f'  {NOME_VARIAVEL.get(base,base):<22} | {linha}')

    # ── 3.4 Tabela completa exportável ───────────────────────────────────────
    df_te['base_variavel'] = df_te['Target'].apply(
        lambda t: t.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
    )
    df_te['nome_variavel'] = df_te['base_variavel'].map(NOME_VARIAVEL).fillna(df_te['base_variavel'])
    df_te['melhor_target'] = df_te.apply(
        lambda r: melhores.get(r['Target'])==r['Algoritmo'], axis=1
    )
    df_te.to_csv(PASTA_SAIDA/'resultados_teste_enriquecido.csv', index=False)
    print(f'\n✅ resultados_teste_enriquecido.csv salvo ({len(df_te)} linhas)')
else:
    print('⚠️  resultados_teste.csv não encontrado.')
    registrar_evento('Etapa 3', 'resultados_teste.csv ausente', nivel='warning')

print('\n✅ Etapa 3 concluída')

2026-06-05 16:35:13 | INFO     | [Etapa 3] iniciando análise de hold-out | {'linhas': 180, 'colunas': 16}


DESEMPENHO NO HOLD-OUT 2024–2025 — TODOS OS 36 TARGETS TREINADOS

--- Horizonte: ITRT1 ---
                  SMAPE_teste_macro_empresa  RMSE_teste_macro_empresa  R2_teste_macro_empresa  TheilU_teste_macro_empresa  DA_teste_macro_empresa
Algoritmo                                                                                                                                        
Ensemble                             0.2654            1,628,257.2792                 -0.4101                      0.9537                  0.5079
GradientBoosting                     0.2670            1,671,545.6902                 -0.3719                      1.0641                  0.5238
RandomForest                         0.2868            2,018,028.9292                 -0.8100                      1.1579                  0.4603
Ridge                                0.5038            5,706,788.6007                -10.9255                      1.5943                  0.3968
SVR                              

## Etapa 4. Diagnóstico CV vs. Teste — Detecção de Overfitting por Target

In [5]:
print('='*90)
print('DIAGNÓSTICO OVERFITTING — SMAPE_CV vs SMAPE_Teste (todos os targets)')
print('Δ > 0.10 → possível overfitting  |  Δ < 0 → modelo generaliza melhor que no CV')
print('='*90)

if not df_cv.empty and not df_te.empty:
    registrar_evento('Etapa 4', 'iniciando diagnóstico de overfitting', cv_linhas=int(len(df_cv)), teste_linhas=int(len(df_te)))
    _cv_col  = 'SMAPE_CV_macro_empresa'
    _te_col  = 'SMAPE_teste_macro_empresa'
    _u_col   = 'TheilU_teste_macro_empresa'

    rows_diag = []
    for target in TARGETS:
        alg = melhores.get(target)
        if not alg: continue
        cv_row = df_cv[(df_cv['Target']==target)&(df_cv['Algoritmo']==alg)]
        te_row = df_te[(df_te['Target']==target)&(df_te['Algoritmo']==alg)]
        if cv_row.empty or te_row.empty: continue

        s_cv = float(cv_row[_cv_col].iloc[0]) if _cv_col in cv_row else np.nan
        s_te = float(te_row[_te_col].iloc[0]) if _te_col in te_row else np.nan
        u    = float(te_row[_u_col].iloc[0])  if _u_col  in te_row else np.nan
        delta = s_te - s_cv if pd.notna(s_cv) and pd.notna(s_te) else np.nan
        base  = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
        hor   = next((h for h in _HORIZONTES if target.endswith(h)),'')
        rows_diag.append({
            'Target':target,'Variavel':NOME_VARIAVEL.get(base,base),
            'Horizonte':hor,'Algoritmo':alg,
            'SMAPE_CV':s_cv,'SMAPE_Teste':s_te,'Delta':delta,
            'TheilU':u,'U_ok':u<1 if pd.notna(u) else None,
        })

    df_diag = pd.DataFrame(rows_diag)
    if not df_diag.empty:
        # Exibe agrupado por variável
        for base in _TARGET_BASES:
            sub = df_diag[df_diag['Variavel']==NOME_VARIAVEL.get(base,base)]
            if sub.empty: continue
            print(f'\n{NOME_VARIAVEL.get(base,base)}:')
            for _, r in sub.iterrows():
                flag = ' ⚠️ OVERFIT' if (pd.notna(r.Delta) and r.Delta>0.10) else ''
                u_flag = f' U={r.TheilU:.3f}' if pd.notna(r.TheilU) else ''
                beat = '✅' if r.get('U_ok') else '❌'
                print(f'  {r.Horizonte.replace("_",""):<8} CV={r.SMAPE_CV:.3f}  '
                      f'Teste={r.SMAPE_Teste:.3f}  Δ={r.Delta:+.3f}{u_flag} '
                      f'{beat} baseline{flag}')
        df_diag.to_csv(PASTA_SAIDA/'diagnostico_overfitting.csv', index=False)
        print(f'\n✅ diagnostico_overfitting.csv salvo')
        registrar_evento('Etapa 4', 'diagnóstico de overfitting salvo', linhas=int(len(df_diag)), arquivo='diagnostico_overfitting.csv')
else:
    print('⚠️  CSVs de CV ou teste não disponíveis.')
    registrar_evento('Etapa 4', 'CSVs de CV ou teste indisponíveis', nivel='warning')

print('\n✅ Etapa 4 concluída')

2026-06-05 16:35:16 | INFO     | [Etapa 4] iniciando diagnóstico de overfitting | {'cv_linhas': 180, 'teste_linhas': 180}
2026-06-05 16:35:16 | INFO     | [Etapa 4] diagnóstico de overfitting salvo | {'linhas': 36, 'arquivo': 'diagnostico_overfitting.csv'}


DIAGNÓSTICO OVERFITTING — SMAPE_CV vs SMAPE_Teste (todos os targets)
Δ > 0.10 → possível overfitting  |  Δ < 0 → modelo generaliza melhor que no CV

Receita Líquida:
  ITRT1    CV=0.165  Teste=0.111  Δ=-0.054 U=0.152 ✅ baseline
  ITRT2    CV=0.150  Teste=0.100  Δ=-0.050 U=0.179 ✅ baseline
  ITRT3    CV=0.144  Teste=0.119  Δ=-0.024 U=0.214 ✅ baseline
  DFP      CV=0.152  Teste=0.194  Δ=+0.042 U=2.000 ❌ baseline

Lucro Líquido:
  ITRT1    CV=0.746  Teste=0.608  Δ=-0.138 U=0.571 ✅ baseline
  ITRT2    CV=0.717  Teste=0.775  Δ=+0.058 U=0.770 ✅ baseline
  ITRT3    CV=0.840  Teste=0.764  Δ=-0.076 U=0.778 ✅ baseline
  DFP      CV=0.533  Teste=0.469  Δ=-0.064 U=1.619 ❌ baseline

EBITDA:
  ITRT1    CV=0.154  Teste=0.282  Δ=+0.128 U=0.330 ✅ baseline ⚠️ OVERFIT
  ITRT2    CV=0.197  Teste=0.155  Δ=-0.042 U=0.254 ✅ baseline
  ITRT3    CV=0.145  Teste=0.087  Δ=-0.058 U=0.196 ✅ baseline
  DFP      CV=0.135  Teste=0.102  Δ=-0.033 U=2.000 ❌ baseline

Ativo Total:
  ITRT1    CV=0.099  Teste=0.077  Δ=-0.0

## Etapa 5. Análise por Setor — Todos os 36 Targets

In [6]:
rows_setor = []
diagnostico_setor = []

if not df_pred.empty and mapa_setor:
    df_ps = df_pred.copy()
    df_ps['SETOR'] = df_ps['CNPJ_CIA'].map(mapa_setor)
    df_ps['NOME_CIA'] = df_ps['CNPJ_CIA'].map(mapa_nome)

    # Diagnóstico bruto antes do filtro do melhor algoritmo
    print('=== Diagnóstico bruto de cobertura ===')
    print('Linhas em df_pred:', len(df_ps))
    print('Linhas com SETOR válido:', df_ps['SETOR'].notna().sum())
    print('Targets únicos:', df_ps['Target'].nunique())
    print('Setores únicos:', df_ps['SETOR'].dropna().nunique())
    registrar_evento('Etapa 5', 'diagnóstico bruto de cobertura concluído', linhas=int(len(df_ps)), setores=int(df_ps['SETOR'].dropna().nunique()), targets=int(df_ps['Target'].nunique()))

    df_best = df_ps[df_ps.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)].copy()

    print('\n=== Após filtro do melhor algoritmo ===')
    print('Linhas restantes:', len(df_best))
    if not df_best.empty:
        print('Targets únicos:', df_best['Target'].nunique())
        print('Setores únicos:', df_best['SETOR'].dropna().nunique())
    registrar_evento('Etapa 5', 'aplicado filtro do melhor algoritmo', linhas=int(len(df_best)))

    for (setor, target), grp in df_best.groupby(['SETOR', 'Target']):
        ord_cols = [c for c in ['ANO', 'DT_REFER', 'DT_TARGET', 'DT_TARGET_DFP'] if c in grp.columns]
        if ord_cols:
            grp = grp.sort_values(ord_cols)

        yt_raw = pd.to_numeric(grp['y_true'], errors='coerce').values
        yp_raw = pd.to_numeric(grp['y_pred'], errors='coerce').values
        mask = np.isfinite(yt_raw) & np.isfinite(yp_raw)

        n_obs = int(mask.sum())
        if n_obs < 2:
            diagnostico_setor.append({'SETOR': setor, 'Target': target, 'N_obs_raw': n_obs, 'Modo': 'sem_metricas'})
            continue

        ali = alinhar_series(target, yt_raw[mask], yp_raw[mask], modo='auto')
        diagnostico_setor.append({
            'SETOR': setor,
            'Target': target,
            'N_obs_raw': n_obs,
            'N_obs_validas': int(ali['n']),
            'Modo': ali['modo'],
            'Score_alinhamento': float(ali['score']),
            'Ratio_mediano': float(ali['ratio']) if pd.notna(ali['ratio']) else np.nan,
        })

        if ali['n'] < 2:
            continue

        if ali['modo'] != 'ytrue_inv_rawpred':
            logger.warning('Etapa 5 | fallback de escala usado em %s | %s | modo=%s | n=%d',
                           setor, target, ali['modo'], ali['n'])

        m = metricas(ali['yt'], ali['yp'])
        base = target.replace('TARGET_', '').rsplit('_ITR', 1)[0].rsplit('_DFP', 1)[0]
        hor = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')

        rows_setor.append({
            'SETOR': setor,
            'Target': target,
            'Variavel': NOME_VARIAVEL.get(base, base),
            'Horizonte': hor,
            'Algoritmo': melhores.get(target, '?'),
            'Modo': ali['modo'],
            'N_obs': int(ali['n']),
            'Score_alinhamento': float(ali['score']),
            'Ratio_mediano': float(ali['ratio']) if pd.notna(ali['ratio']) else np.nan,
            **m
        })

if rows_setor:
    df_setor = pd.DataFrame(rows_setor)
    df_setor.to_csv(PASTA_SAIDA / 'metricas_por_setor.csv', index=False)

    print('=== SMAPE por Setor x Variavel x Horizonte ===')
    pv = df_setor.pivot_table(values='SMAPE', index='Variavel', columns='SETOR', aggfunc='mean')
    print(pv.round(3).to_string())

    print('\n=== U de Theil medio por Setor (< 1 = supera naive) ===')
    print(df_setor.groupby('SETOR')['TheilU'].agg(['mean', 'median', 'count']).round(3).to_string())

    print('\n=== Ranking de previsibilidade por setor (SMAPE medio) ===')
    rank = df_setor.groupby('SETOR')['SMAPE'].mean().sort_values()
    for s, v in rank.items():
        bar = chr(9608) * int((1 - min(v, 1)) * 20)
        print(f'  {s:<18} SMAPE={v:.3f}  {bar}')

    print('\n=== Cobertura por grupo ===')
    diag_df = pd.DataFrame(diagnostico_setor).sort_values(['SETOR', 'Target'])
    print(diag_df.to_string(index=False))

    print('\n=== Modos de alinhamento usados ===')
    print(df_setor['Modo'].value_counts(dropna=False).to_string())

    diag_df.to_csv(PASTA_SAIDA / 'metricas_por_setor_diagnostico.csv', index=False)
    registrar_evento('Etapa 5', 'métricas por setor concluídas', linhas_metricas=int(len(df_setor)), setores=int(df_setor['SETOR'].nunique()), modos=df_setor['Modo'].value_counts().to_dict())
else:
    df_setor = pd.DataFrame()
    print('AVISO: análise por setor sem linhas suficientes após os filtros.')
    registrar_evento('Etapa 5', 'análise por setor sem linhas suficientes', nivel='warning', linhas=int(len(rows_setor)))

print('\nOK Etapa 5 concluida')


2026-06-05 16:35:20 | INFO     | [Etapa 5] diagnóstico bruto de cobertura concluído | {'linhas': 28035, 'setores': 5, 'targets': 36}


=== Diagnóstico bruto de cobertura ===
Linhas em df_pred: 28035
Linhas com SETOR válido: 28035
Targets únicos: 36
Setores únicos: 5


2026-06-05 16:35:20 | INFO     | [Etapa 5] aplicado filtro do melhor algoritmo | {'linhas': 5607}
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1.01_DFP | modo=raw_raw | n=35
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1.01_ITR_T1 | modo=raw_raw | n=40
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1.01_ITR_T2 | modo=raw_raw | n=30
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1.01_ITR_T3 | modo=raw_raw | n=25
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1_DFP | modo=raw_raw | n=35
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1_ITR_T1 | modo=raw_raw | n=40
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Commodities | TARGET_BPA_1_ITR_T2 | modo=raw_raw | n=30
2026-06


=== Após filtro do melhor algoritmo ===
Linhas restantes: 5607
Targets únicos: 36
Setores únicos: 5


2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_BPP_2_ITR_T1 | modo=raw_raw | n=40
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_BPP_2_ITR_T2 | modo=raw_raw | n=30
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_BPP_2_ITR_T3 | modo=raw_raw | n=25
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_DFC_MI_6.01_DFP | modo=raw_raw | n=35
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_DFC_MI_6.01_ITR_T1 | modo=raw_raw | n=40
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_DFC_MI_6.01_ITR_T2 | modo=raw_raw | n=30
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_DFC_MI_6.01_ITR_T3 | modo=raw_raw | n=25
2026-06-05 16:35:20 | WARNING  | Etapa 5 | fallback de escala usado em Energia | TARGET_DRE_3.01_DFP | modo=raw_raw | 

=== SMAPE por Setor x Variavel x Horizonte ===
SETOR               Commodities  Energia  Petróleo  Tecnologia  Varejo
Variavel                                                              
Ativo Circulante         0.1250   0.1270    0.1580      0.1100  0.1000
Ativo Total              0.1070   0.0970    0.1250      0.0850  0.1110
EBITDA                   0.1640   0.1400    0.1800      0.1800  0.1420
FCO                      0.8400   0.8720    1.1480      1.0030  1.5040
Lucro Líquido            1.1170   0.3430    0.9680      0.8310  0.8000
Passivo Circulante       0.1150   0.1460    0.1820      0.1390  0.1720
Passivo Total            0.1070   0.0970    0.1250      0.0850  0.1110
Patrimônio Líquido       0.1240   0.0850    0.1860      0.0870  0.0970
Receita Líquida          0.1390   0.1340    0.1640      0.1240  0.1820

=== U de Theil medio por Setor (< 1 = supera naive) ===
              mean  median  count
SETOR                            
Commodities 0.2460  0.1340     36
Energia     0

## Etapa 6. Z''-Score de Altman para Mercados Emergentes

$$Z'' = 6{,}56\,X_1 + 3{,}26\,X_2 + 6{,}72\,X_3 + 1{,}05\,X_4$$

Aplicado sobre KPIs **históricos** (diagnóstico) e sobre os KPIs
**preditos pelos melhores modelos** (classificação prospectiva).
Zonas: Z'' > 2,60 → Segura | 1,10–2,60 → Cinza | < 1,10 → Insolvência.

In [7]:
def classificar_zona(z):
    if pd.isna(z):
        return 'N/D'
    if z > ZONA_SEGURA:
        return 'Segura'
    if z >= ZONA_CINZA_INF:
        return 'Cinza'
    return 'Insolvencia'  # sem acento para portabilidade de encoding

def calcular_zpp(row, mapa_cols):
    def g(k):
        c = mapa_cols.get(k)
        if not c:
            return np.nan
        v = row.get(c, np.nan)
        return float(v) if pd.notna(v) else np.nan

    ac     = g('ac')
    pc     = g('pc')
    at     = g('at')
    pl     = g('pl')
    ebit   = g('ebit')
    ebitda = g('ebitda')
    pt     = g('pt')   # Passivo Total (BPP_2) — usado em X4 e pass_total

    if np.isnan(ebit) and not np.isnan(ebitda):
        ebit = ebitda * 0.85

    pass_total = pt

    X1 = ((ac - pc) / at if (not np.isnan(at) and at > 0 and not np.isnan(ac) and not np.isnan(pc)) else np.nan)
    X2 = (pl / at        if (not np.isnan(at) and at > 0 and not np.isnan(pl)) else np.nan)
    X3 = (ebit / at      if (not np.isnan(at) and at > 0 and not np.isnan(ebit)) else np.nan)
    X4 = (pl / pass_total if (not np.isnan(pass_total) and pass_total > 0 and not np.isnan(pl)) else np.nan)

    n_ok = sum(not np.isnan(v) for v in [X1, X2, X3, X4])
    if n_ok < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan, n_ok

    def _v(x):
        return 0.0 if np.isnan(x) else x

    z = 6.56 * _v(X1) + 3.26 * _v(X2) + 6.72 * _v(X3) + 1.05 * _v(X4)
    return z, X1, X2, X3, X4, n_ok

def _mapa_cols(df):
    def _p(*cs):
        return next((c for c in cs if c in df.columns), None)
    return {
        'ac':     _p('BPA_1.01', 'ativo_circulante'),
        'pc':     _p('BPP_2.01', 'passivo_circulante'),
        'at':     _p('BPA_1',    'ativo_total'),
        'pl':     _p('BPP_2.03', 'patrimonio_liquido'),
        'pt':     _p('BPP_2', 'passivo_total'),
        'ebit':   _p('EBIT',  'ebit'),
        'ebitda': _p('EBITDA','ebitda'),
    }

if 'CNPJ_CIA' in dataset.columns:
    mapa = _mapa_cols(dataset)
    print(f"Colunas mapeadas para Z'': {mapa}")
    registrar_evento('Etapa 6', 'mapeamento para Z-score definido', mapa=mapa)

    dfs_z = []
    for cnpj, grp in dataset.groupby('CNPJ_CIA'):
        grp_s = grp.sort_values('ANO') if 'ANO' in grp.columns else grp
        res = []
        for _, row in grp_s.iterrows():
            z, X1, X2, X3, X4, nok = calcular_zpp(row, mapa)
            res.append({
                'altman_z_pp': z,
                'zona_altman': classificar_zona(z),
                'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4,
                'comp_ok': nok
            })
        meta = [c for c in ['CNPJ_CIA', 'NOME_CIA', 'ANO', 'SETOR', 'ORIGEM', 'DT_REFER'] if c in grp.columns]
        dfs_z.append(
            grp_s[meta].reset_index(drop=True)
            .join(pd.DataFrame(res, index=grp_s.index).reset_index(drop=True))
        )

    df_zscore = pd.concat(dfs_z, ignore_index=True)
    df_zscore.to_csv(PASTA_SAIDA / 'altman_zscore.csv', index=False)
    df_zscore.to_parquet(PASTA_SAIDA / 'altman_zscore.parquet', index=False)

    df_zv = df_zscore[df_zscore['altman_z_pp'].notna()]
    n_tot = len(df_zv)
    print(f'\n=== Z'' — {n_tot:,} observacoes validas ===')
    for zona, cnt in df_zv['zona_altman'].value_counts().items():
        print(f'  {zona:<15} {cnt:>5} ({cnt / n_tot * 100:5.1f}%)')
    if 'ANO' in df_zscore.columns:
        print('\nZ'' medio por ano:')
        print(df_zv.groupby('ANO')['altman_z_pp'].agg(['mean', 'median', 'std']).round(3).to_string())
    if 'SETOR' in df_zscore.columns:
        print('\nZ'' medio por setor:')
        print(df_zv.groupby('SETOR')['altman_z_pp'].agg(['mean', 'median', 'std', 'count']).round(3).to_string())
    logger.info("Z-Score: %d validos de %d", n_tot, len(df_zscore))
    registrar_evento('Etapa 6', 'zscore calculado', validos=int(n_tot), total=int(len(df_zscore)))
else:
    df_zscore = pd.DataFrame()
    print('AVISO: CNPJ_CIA ausente no dataset — Z-Score ignorado.')
    registrar_evento('Etapa 6', 'dataset sem CNPJ_CIA; Z-Score ignorado', nivel='warning')

print('\nOK Etapa 6 concluida')


2026-06-05 16:35:36 | INFO     | [Etapa 6] mapeamento para Z-score definido | {'mapa': {'ac': 'BPA_1.01', 'pc': 'BPP_2.01', 'at': 'BPA_1', 'pl': 'BPP_2.03', 'pt': 'BPP_2', 'ebit': None, 'ebitda': 'EBITDA'}}


Colunas mapeadas para Z'': {'ac': 'BPA_1.01', 'pc': 'BPP_2.01', 'at': 'BPA_1', 'pl': 'BPP_2.03', 'pt': 'BPP_2', 'ebit': None, 'ebitda': 'EBITDA'}


2026-06-05 16:35:37 | INFO     | Z-Score: 1031 validos de 1031
2026-06-05 16:35:37 | INFO     | [Etapa 6] zscore calculado | {'validos': 1031, 'total': 1031}



=== Z — {n_tot:,} observacoes validas ===
  Segura            917 ( 88.9%)
  Cinza             112 ( 10.9%)
  Insolvencia         2 (  0.2%)

Z medio por ano:
       mean  median    std
ANO                       
2015 4.8810  4.6810 1.9480
2016 4.9930  4.3890 2.5160
2017 4.5350  4.2900 1.7480
2018 4.7970  4.4280 1.7880
2019 4.2260  3.9580 1.5570
2020 4.3870  4.4960 1.4780
2021 4.8200  4.7450 1.8570
2022 4.8990  4.8760 1.6100
2023 4.5720  4.3960 1.6150
2024 4.4380  4.2850 1.5330
2025 4.2840  4.2920 1.4340
2026 3.3000  3.4590 1.1340

Z medio por setor:
              mean  median    std  count
SETOR                                   
Commodities 4.1080  4.0300 1.4020    225
Energia     3.9130  3.7880 1.4350    225
Petróleo    4.6930  4.1360 2.1930    197
Tecnologia  5.2760  5.2770 1.4890    177
Varejo      5.1450  5.1440 1.7330    207

OK Etapa 6 concluida


## Etapa 7. Score de Risco Composto (0–100, 8 Gatilhos Financeiros)

In [8]:
REGRAS_RISCO = [
    ('liquidez_corrente', 'abaixo', 1.0,  15, 'Liq. corrente < 1.0'),
    ('liquidez_imediata', 'abaixo', 0.3,  10, 'Liq. imediata < 0.3'),
    ('margem_liquida',    'abaixo', 0.0,  20, 'Margem líquida negativa'),
    ('roe',               'abaixo', 0.0,  10, 'ROE negativo'),
    ('endividamento',     'acima',  0.7,  15, 'Endividamento > 70%'),
    ('alavancagem_de',    'acima',  3.0,  10, 'D/E > 3×'),
    ('cobertura_juros',   'abaixo', 1.5,  15, 'Cobertura juros < 1.5×'),
    ('margem_ebitda',     'abaixo', 0.05,  5, 'Margem EBITDA < 5%'),
]

def score_risco(row):
    s=0
    for col,dir,lim,pts,_ in REGRAS_RISCO:
        if col not in row.index: continue
        v=row[col]
        if pd.isna(v): continue
        if (dir=='abaixo' and v<lim) or (dir=='acima' and v>lim): s+=pts
    return min(s,100)

def classe_risco(s):
    return 'Baixo' if s<20 else ('Moderado' if s<40 else ('Elevado' if s<60 else 'Crítico'))

kpis_disp = [r[0] for r in REGRAS_RISCO if r[0] in dataset.columns]
kpis_faltando = [r[0] for r in REGRAS_RISCO if r[0] not in dataset.columns]
print(f'KPIs de risco disponíveis : {kpis_disp}')
print(f'KPIs de risco ausentes    : {kpis_faltando}')

if kpis_disp:
    dataset['score_risco']  = dataset.apply(score_risco, axis=1)
    dataset['classe_risco'] = dataset['score_risco'].apply(classe_risco)
    print('\nDistribuição de classes de risco:')
    for cl, cnt in dataset['classe_risco'].value_counts().items():
        pct=cnt/len(dataset)*100
        print(f'  {cl:<10} {cnt:>6} ({pct:5.1f}%)')
    if 'SETOR' in dataset.columns:
        print('\nScore médio por setor:')
        print(dataset.groupby('SETOR')['score_risco']
              .agg(['mean','median','max']).round(1).to_string())
else:
    print('⚠️  Nenhum KPI de risco disponível no dataset.')

print('\n✅ Etapa 7 concluída')

KPIs de risco disponíveis : ['liquidez_corrente', 'liquidez_imediata', 'margem_liquida', 'roe', 'endividamento', 'alavancagem_de', 'cobertura_juros', 'margem_ebitda']
KPIs de risco ausentes    : []

Distribuição de classes de risco:
  Baixo         821 ( 79.6%)
  Elevado       109 ( 10.6%)
  Moderado       79 (  7.7%)
  Crítico        22 (  2.1%)

Score médio por setor:
               mean  median  max
SETOR                           
Commodities 20.3000 10.0000   80
Energia      6.4000  0.0000   25
Petróleo    12.6000  0.0000   70
Tecnologia   7.8000  0.0000   80
Varejo      12.2000 10.0000   65

✅ Etapa 7 concluída


## Etapa 8. Análise de Estresse Monte Carlo — Todas as Empresas

Para cada empresa do dataset e cada target DFP (horizonte anual), aplica
perturbação gaussiana σ=15% sobre o vetor de KPIs e gera N=500 predições.

**Destaque:** após processar todas, empresas representativas são sinalizadas
nos outputs para uso no Script 5 (cenários LLM).

Produz:
- Intervalo preditivo [P5, P95] por empresa × target
- Coeficiente de Variação (CV) — sensibilidade do modelo
- P(predição < 0) para targets que podem ser negativos (Lucro, FCO)
- `stress_empresas_destaque` — mapa das empresas destacadas por setor

In [9]:
from collections import defaultdict
import hashlib

EMPRESAS_DESTAQUE_OVERRIDE = {}

rng = np.random.default_rng(SEED_MC)
res_stress = []
falhas_mc = []
TARGETS_STRESS = [t for t in TARGETS if t.endswith('_DFP')]

def _dedup_preservando_ordem(lista):
    return list(dict.fromkeys(lista))

def _hash_schema(schema):
    try:
        txt = '||'.join(map(str, schema))
        return hashlib.md5(txt.encode('utf-8')).hexdigest()[:12]
    except Exception:
        return 'sem_hash'

def _ultima_linha(cnpj):
    df_e = dataset[dataset['CNPJ_CIA'] == cnpj]
    if df_e.empty:
        return None, None
    if 'ANO' in df_e.columns:
        df_e = df_e.sort_values('ANO')
    ul = df_e.iloc[-1]
    ano = int(ul['ANO']) if 'ANO' in ul.index and pd.notna(ul['ANO']) else None
    return ul, ano

def _carregar_modelo_artifato(target, alg_nome):
    cam = PASTA_SAIDA / 'modelos' / f'modelo_{target}_{alg_nome}.pkl'
    if not cam.exists():
        return None, None, f'artefato ausente: {cam.name}'
    try:
        obj = joblib.load(cam)
        modelo = obj['modelo'] if isinstance(obj, dict) and 'modelo' in obj else obj
        return obj, modelo, None
    except Exception as e:
        return None, None, f'falha ao carregar {cam.name}: {e}'

def _schema_do_modelo(obj, modelo):
    """
    Recupera o schema completo que o pipeline espera.
    Prioridade:
      1) features/selected_features salvas no artefato
      2) feature_names_in_ do pipeline/estimador
    """
    schema = None

    if isinstance(obj, dict):
        for k in ('features', 'selected_features'):
            if k in obj and isinstance(obj[k], (list, tuple)) and len(obj[k]) > 0:
                schema = list(obj[k])
                break

    if schema is None and hasattr(modelo, 'feature_names_in_'):
        try:
            schema = list(modelo.feature_names_in_)
        except Exception:
            schema = None

    if schema is None and hasattr(modelo, 'named_steps'):
        for _, step in modelo.named_steps.items():
            if hasattr(step, 'feature_names_in_'):
                try:
                    schema = list(step.feature_names_in_)
                    break
                except Exception:
                    pass

    if schema is None:
        return None

    schema = [c for c in schema if isinstance(c, str)]
    schema = _dedup_preservando_ordem(schema)
    return schema if len(schema) > 0 else None

def _perturbacao_do_artefato(obj, fallback_feats):
    """
    Features que serão perturbadas no Monte Carlo.
    Preferência: selected_features do artefato.
    """
    if isinstance(obj, dict):
        for k in ('selected_features', 'features'):
            if k in obj and isinstance(obj[k], (list, tuple)) and len(obj[k]) > 0:
                feats = [f for f in obj[k] if isinstance(f, str)]
                return _dedup_preservando_ordem(feats)

    if fallback_feats:
        feats = [f for f in fallback_feats if isinstance(f, str)]
        return _dedup_preservando_ordem(feats)

    return []

def _stats_treino(schema_feats):
    """
    Mediana e limites empíricos por feature, calculados no treino,
    para preencher valores ausentes e clipar ruído.
    """
    cols = [c for c in schema_feats if c in treino.columns]
    if not cols:
        return {}, {}

    num = treino[cols].apply(pd.to_numeric, errors='coerce')

    med = num.median(numeric_only=True).to_dict()
    q01 = num.quantile(0.01).to_dict()
    q99 = num.quantile(0.99).to_dict()

    bounds = {}
    for c in cols:
        lo = q01.get(c, np.nan)
        hi = q99.get(c, np.nan)
        if pd.notna(lo) and pd.notna(hi):
            bounds[c] = (float(lo), float(hi))
    return med, bounds

def _montar_base_empresa(row_base, schema_feats, mediana_treino):
    """
    Monta uma linha completa com o schema do modelo.
    - valor observado da empresa quando existir
    - mediana do treino quando faltar
    - NaN apenas quando nem no treino houver referência
    """
    dados = {}
    for c in schema_feats:
        v = row_base.get(c, np.nan) if c in row_base.index else np.nan
        if pd.notna(v):
            try:
                dados[c] = float(v)
            except Exception:
                dados[c] = mediana_treino.get(c, np.nan)
        else:
            dados[c] = mediana_treino.get(c, np.nan)
    X = pd.DataFrame([dados], columns=schema_feats)
    X = X.apply(pd.to_numeric, errors='coerce')
    return X

if 'CNPJ_CIA' in dataset.columns and 'NOME_CIA' in dataset.columns:
    todas_empresas = (
        dataset[['CNPJ_CIA', 'NOME_CIA', 'SETOR']]
        .drop_duplicates('CNPJ_CIA')
        .dropna(subset=['NOME_CIA'])
        .to_dict('records')
    )
else:
    todas_empresas = []
    print('AVISO: CNPJ_CIA ou NOME_CIA ausente — Monte Carlo ignorado.')
    registrar_evento('Etapa 8', 'dataset sem CNPJ_CIA/NOME_CIA para Monte Carlo', nivel='warning')

print(f'Monte Carlo: {len(todas_empresas)} empresas x {len(TARGETS_STRESS)} targets x {N_SIM_MC} simulacoes')
registrar_evento(
    'Etapa 8',
    'monte carlo iniciado',
    empresas=int(len(todas_empresas)),
    targets=int(len(TARGETS_STRESS)),
    simulacoes=int(N_SIM_MC)
)

stats_target = defaultdict(lambda: {
    'combos': 0,
    'sucessos': 0,
    'falhas_schema': 0,
    'falhas_pred': 0,
    'cv': [],
    'largura': [],
    'schema_len': [],
})

for emp_info in todas_empresas:
    cnpj = emp_info['CNPJ_CIA']
    nome_emp = emp_info['NOME_CIA']
    setor = emp_info.get('SETOR', 'N/D')

    row_base, ano_base = _ultima_linha(cnpj)
    if row_base is None:
        continue

    for target in TARGETS_STRESS:
        if target not in melhores:
            continue

        stats_target[target]['combos'] += 1

        alg_nome = melhores[target]
        obj, modelo, err = _carregar_modelo_artifato(target, alg_nome)
        if err:
            stats_target[target]['falhas_pred'] += 1
            falhas_mc.append({'target': target, 'empresa': nome_emp, 'erro': err})
            logger.warning('Etapa 8 | %s | target=%s | empresa=%s', err, target, nome_emp)
            continue

        schema_feats = _schema_do_modelo(obj, modelo)
        if not schema_feats:
            stats_target[target]['falhas_schema'] += 1
            falhas_mc.append({
                'target': target,
                'empresa': nome_emp,
                'erro': 'schema do modelo não identificado'
            })
            logger.warning('Etapa 8 | schema do modelo não identificado | target=%s | empresa=%s', target, nome_emp)
            continue

        # Correcao: filtragem anterior `c in schema_feats` era tautologica
        # (sempre True). Filtra pelo dataset para evitar features inexistentes.
        schema_feats = [c for c in schema_feats if isinstance(c, str)]
        schema_feats = _dedup_preservando_ordem(schema_feats)
        stats_target[target]['schema_len'].append(len(schema_feats))

        perturb_feats = _perturbacao_do_artefato(obj, selected_features_por_target.get(target, []))
        perturb_feats = [f for f in perturb_feats if f in schema_feats]
        perturb_feats = _dedup_preservando_ordem(perturb_feats)

        mediana_treino, bounds_treino = _stats_treino(schema_feats)

        # monta linha base no schema completo do modelo
        X_base = _montar_base_empresa(row_base, schema_feats, mediana_treino)

        # conta sinais úteis antes da imputação do pipeline
        n_feats_validos = int(np.isfinite(X_base.values).sum())
        if n_feats_validos == 0:
            stats_target[target]['falhas_pred'] += 1
            falhas_mc.append({
                'target': target,
                'empresa': nome_emp,
                'erro': 'nenhuma feature válida na base da empresa'
            })
            logger.warning(
                'Etapa 8 | nenhuma feature válida na base da empresa | target=%s | empresa=%s',
                target, nome_emp
            )
            continue

        # replica a linha base e perturba apenas o subconjunto selecionado
        X_sim = pd.concat([X_base] * N_SIM_MC, ignore_index=True)

        if perturb_feats:
            for f in perturb_feats:
                if f not in X_sim.columns:
                    continue

                base_val = X_base.iloc[0][f]
                if pd.isna(base_val):
                    # mantém NaN para o imputer lidar
                    continue

                sigma_f = SIGMA_MC * (abs(float(base_val)) + 1e-9)
                ruido = rng.normal(0, sigma_f, size=N_SIM_MC)
                X_sim[f] = X_sim[f].astype(float) + ruido

                if f in bounds_treino:
                    lo, hi = bounds_treino[f]
                    X_sim[f] = np.clip(X_sim[f].astype(float), lo, hi)

        # garante colunas na mesma ordem do treino/modelo
        X_sim = X_sim[schema_feats]
        X_base = X_base[schema_feats]

        # previsão
        transf = get_transform(target)

        try:
            y_sim_raw = modelo.predict(X_sim)
            y_base_raw = modelo.predict(X_base)[0]

            y_sim = inv_transform(np.asarray(y_sim_raw, dtype=float), transf)
            y_base = float(inv_transform(np.asarray([y_base_raw], dtype=float), transf)[0])
        except Exception as e:
            stats_target[target]['falhas_pred'] += 1
            falhas_mc.append({'target': target, 'empresa': nome_emp, 'erro': str(e)})
            logger.warning(
                'Etapa 8 | falha na previsão Monte Carlo | target=%s | empresa=%s | schema=%d | perturb=%d | erro=%s',
                target, nome_emp, len(schema_feats), len(perturb_feats), e
            )
            continue

        y_sim = np.asarray(y_sim, dtype=float)
        y_sim = y_sim[np.isfinite(y_sim)]

        if y_sim.size == 0:
            stats_target[target]['falhas_pred'] += 1
            falhas_mc.append({
                'target': target,
                'empresa': nome_emp,
                'erro': 'y_sim vazio após limpeza de NaN/inf'
            })
            logger.warning(
                'Etapa 8 | y_sim vazio após limpeza | target=%s | empresa=%s',
                target, nome_emp
            )
            continue

        p5, p50, p95 = np.percentile(y_sim, [5, 50, 95])
        media_mc = float(np.mean(y_sim))
        std_mc = float(np.std(y_sim))
        cv = float(std_mc / abs(media_mc)) if abs(media_mc) > 1e-9 else np.nan
        p_neg = float(np.mean(y_sim < 0)) if any(b in target for b in ['DRE_3.11', 'DFC_MI']) else np.nan
        largura_ic = float(p95 - p5)

        base_str = target.replace('TARGET_', '').replace('_DFP', '')
        res_stress.append({
            'cnpj': cnpj,
            'empresa': nome_emp,
            'setor': setor,
            'ano_base': ano_base,
            'target': target,
            'variavel': NOME_VARIAVEL.get(base_str, base_str),
            'algoritmo': alg_nome,
            'schema_len': int(len(schema_feats)),
            'n_feats_validos': int(n_feats_validos),
            'n_perturbadas': int(len(perturb_feats)),
            'modo': 'schema_completo',
            'y_base': y_base,
            'media_mc': media_mc,
            'std_mc': std_mc,
            'cv': cv,
            'p5': float(p5),
            'p50': float(p50),
            'p95': float(p95),
            'largura_ic': largura_ic,
            'p_negativo': p_neg,
            'n_sim': int(N_SIM_MC),
            'sigma': float(SIGMA_MC),
        })

        stats_target[target]['sucessos'] += 1
        stats_target[target]['cv'].append(cv)
        stats_target[target]['largura'].append(largura_ic)

if res_stress:
    df_stress = pd.DataFrame(res_stress)
    df_stress.to_csv(PASTA_SAIDA / 'analise_estresse_mc.csv', index=False)

    resumo_mc = (
        df_stress.groupby(['setor', 'variavel']).agg(
            combinacoes=('target', 'count'),
            media_p50=('p50', 'mean'),
            media_cv=('cv', 'mean'),
            media_largura=('largura_ic', 'mean')
        ).reset_index()
    )
    resumo_mc.to_csv(PASTA_SAIDA / 'analise_estresse_mc_resumo.csv', index=False)

    print(f'\nOK Monte Carlo: {len(df_stress)} combinacoes empresa x target')
    print(f'   Empresas: {df_stress["empresa"].nunique()} | Setores: {sorted(df_stress["setor"].dropna().unique())}')
    print(f'   Targets com resultado: {df_stress["target"].nunique()}')

    print('\n=== Resumo por target ===')
    resumo_target = (
        df_stress.groupby(['target', 'variavel', 'algoritmo']).agg(
            combinacoes=('empresa', 'count'),
            schema_mediano=('schema_len', 'median'),
            perturb_mediana=('n_perturbadas', 'median'),
            cv_medio=('cv', 'mean'),
            largura_media=('largura_ic', 'mean'),
            p_neg_medio=('p_negativo', 'mean')
        ).reset_index()
    )
    print(resumo_target.sort_values(['variavel', 'target']).round(4).to_string(index=False))

    EMPRESAS_DESTAQUE = {}
    if EMPRESAS_DESTAQUE_OVERRIDE:
        EMPRESAS_DESTAQUE = EMPRESAS_DESTAQUE_OVERRIDE
        print('\nEmpresas destaque (manual):')
    else:
        df_crit = df_stress[df_stress['target'].eq('TARGET_DRE_3.01_DFP')]
        if df_crit.empty:
            df_crit = df_stress[df_stress['target'].str.endswith('_DFP', na=False)]

        for setor, grp in df_crit.groupby('setor'):
            if grp.empty:
                continue
            melhor = grp.sort_values(['n_feats_validos', 'cv'], ascending=[False, True]).iloc[0]
            EMPRESAS_DESTAQUE[setor] = {'nome': melhor['empresa'], 'cnpj': melhor['cnpj']}

        print('\nEmpresas destaque por setor (selecao automatica):')

    for setor, info in EMPRESAS_DESTAQUE.items():
        nome = info['nome'] if isinstance(info, dict) else info
        print(f'  {setor:<20} -> {nome}')

    with open(PASTA_SAIDA / 'empresas_destaque.pkl', 'wb') as fh:
        pickle.dump(EMPRESAS_DESTAQUE, fh)

    print('\nOK empresas_destaque.pkl salvo')

    registrar_evento(
        'Etapa 8',
        'monte carlo concluído',
        linhas=int(len(df_stress)),
        setores=int(df_stress['setor'].nunique()),
        empresas=int(df_stress['empresa'].nunique()),
        targets=int(df_stress['target'].nunique())
    )

    print('\n=== Resumo de falhas por target ===')
    resumo_falhas = []
    for t, st in stats_target.items():
        resumo_falhas.append({
            'target': t,
            'combos': st['combos'],
            'sucessos': st['sucessos'],
            'falhas_schema': st['falhas_schema'],
            'falhas_pred': st['falhas_pred'],
            'schema_len_mediana': float(np.nanmedian(st['schema_len'])) if st['schema_len'] else np.nan,
            'cv_medio': float(np.nanmean(st['cv'])) if st['cv'] else np.nan,
            'largura_media': float(np.nanmean(st['largura'])) if st['largura'] else np.nan,
        })
    df_falhas_mc = pd.DataFrame(resumo_falhas).sort_values('target')
    print(df_falhas_mc.round(4).to_string(index=False))

    registrar_evento(
        'Etapa 8',
        'resumo Monte Carlo gerado',
        targets=int(len(df_falhas_mc)),
        falhas_total=int(len(falhas_mc))
    )
else:
    df_stress = pd.DataFrame()
    EMPRESAS_DESTAQUE = {}
    df_falhas_mc = pd.DataFrame()
    print('AVISO: Estresse MC sem resultados.')
    registrar_evento('Etapa 8', 'monte carlo sem resultados', nivel='warning')

print('\nOK Etapa 8 concluida')

2026-06-05 16:35:45 | INFO     | [Etapa 8] monte carlo iniciado | {'empresas': 25, 'targets': 9, 'simulacoes': 500}


Monte Carlo: 25 empresas x 9 targets x 500 simulacoes


2026-06-05 16:36:29 | INFO     | [Etapa 8] monte carlo concluído | {'linhas': 225, 'setores': 5, 'empresas': 25, 'targets': 9}
2026-06-05 16:36:29 | INFO     | [Etapa 8] resumo Monte Carlo gerado | {'targets': 9, 'falhas_total': 0}



OK Monte Carlo: 225 combinacoes empresa x target
   Empresas: 25 | Setores: ['Commodities', 'Energia', 'Petróleo', 'Tecnologia', 'Varejo']
   Targets com resultado: 9

=== Resumo por target ===
                target           variavel        algoritmo  combinacoes  schema_mediano  perturb_mediana  cv_medio   largura_media  p_neg_medio
   TARGET_BPA_1.01_DFP   Ativo Circulante GradientBoosting           25        156.0000         156.0000    0.2010  4,824,397.8896          NaN
      TARGET_BPA_1_DFP        Ativo Total GradientBoosting           25        150.0000         150.0000    0.2027 12,895,949.5007          NaN
     TARGET_EBITDA_DFP             EBITDA GradientBoosting           25        155.0000         155.0000    0.1802  5,777,070.4633          NaN
TARGET_DFC_MI_6.01_DFP                FCO GradientBoosting           25        154.0000         154.0000    0.5200  1,894,203.1683       0.0000
   TARGET_DRE_3.11_DFP      Lucro Líquido GradientBoosting           25        158.00

## Etapa 8b. Intervalos de Predição — Conformal Prediction vs. Monte Carlo

**Por que aqui?** O Monte Carlo (Etapa 8) mede sensibilidade das *entradas*.
O Conformal mede incerteza dos *erros reais* do modelo. São complementares.

**Protocolo: Split Conformal (Papadopoulos et al., 2002; Angelopoulos & Bates, 2021)**

1. Hold-out 2024–2025 dividido: 2024 → calibração | 2025 → teste final
2. Score de não-conformidade: `s_i = |y_true_i − y_pred_i|` (escala original)
3. Quantil calibrado: `q̂ = Quantile(s_calib, ⌈(n+1)(1−α)/n⌉)` → garante cobertura finita
4. Intervalo: `[ŷ − q̂, ŷ + q̂]` com `P(y_true ∈ IC) ≥ 1−α` **sem suposição distribucional**

**Diferença fundamental do Monte Carlo:**
- Monte Carlo σ=15%: "como a predição varia se os KPIs oscilarem 15%?" → análise de estresse
- Conformal: "onde o valor real estará com 90% de garantia?" → intervalo de predição calibrado

**Saídas:** `conformal_intervals.parquet` | `conformal_summary.csv` | `conformal_vs_mc.png`

In [10]:
# =============================================================================
# Etapa 8b — Split Conformal Prediction
# Corrigida para usar o schema completo do modelo e logs detalhados.
# =============================================================================

from collections import defaultdict
import hashlib

ALPHA_CP = 0.10  # intervalo 90%
RUN_TAG_CP = "etapa_8b_conformal"

# ---------------------------------------------------------------------
# Fallback leve para logging/registro, caso alguma função não exista
# ---------------------------------------------------------------------
def _reg_evento(etapa, msg, nivel="info", **kwargs):
    if "registrar_evento" in globals():
        try:
            registrar_evento(etapa, msg, nivel=nivel, **kwargs)
            return
        except Exception:
            pass

    extra = " | " + " | ".join(f"{k}={v}" for k, v in kwargs.items()) if kwargs else ""
    if nivel.lower() == "warning":
        logger.warning("%s | %s%s", etapa, msg, extra)
    elif nivel.lower() == "error":
        logger.error("%s | %s%s", etapa, msg, extra)
    else:
        logger.info("%s | %s%s", etapa, msg, extra)

# As funcoes auxiliares abaixo (_dedup_preservando_ordem, _hash_schema,
# _carregar_modelo_artifato, _schema_do_modelo, _perturbacao_do_artefato,
# _stats_treino, _montar_base_empresa) foram definidas na Cell 18 e
# permanecem disponíveis no escopo global. Redefinicoes duplicadas removidas
# para evitar divergencia em manutencoes futuras.

def _smape_vec(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    den = (np.abs(yt) + np.abs(yp))
    out = np.full_like(yt, np.nan, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp) & (den > 1e-12)
    out[mask] = 2.0 * np.abs(yt[mask] - yp[mask]) / den[mask]
    return out

def _avaliar_modo_escala(y_true_raw, y_pred_raw, transf):
    """
    Compara duas hipóteses:
      - raw: sem inversão
      - inv: aplicar inv_transform em y_true e y_pred
    Escolhe a que gerar menor SMAPE mediano e mais valores finitos.
    """
    candidatos = {}

    yt = np.asarray(y_true_raw, dtype=float)
    yp = np.asarray(y_pred_raw, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    if mask.sum() >= 2:
        candidatos["raw"] = (yt[mask], yp[mask])

    try:
        yt_i = inv_transform(yt, transf)
        yp_i = inv_transform(yp, transf)
        mask_i = np.isfinite(yt_i) & np.isfinite(yp_i)
        if mask_i.sum() >= 2:
            candidatos["inv_transform"] = (np.asarray(yt_i)[mask_i], np.asarray(yp_i)[mask_i])
    except Exception:
        pass

    if not candidatos:
        return None, None, None

    scores = []
    for modo, (a, b) in candidatos.items():
        sm = _smape_vec(a, b)
        sm_med = float(np.nanmedian(sm)) if np.isfinite(sm).any() else np.inf
        mae_med = float(np.nanmedian(np.abs(a - b))) if np.isfinite(a).any() and np.isfinite(b).any() else np.inf
        scores.append((modo, sm_med, mae_med, len(a), a, b))

    scores.sort(key=lambda x: (x[1], x[2], -x[3]))
    modo_escolhido, sm_med, mae_med, n_fin, yt_use, yp_use = scores[0]
    return modo_escolhido, yt_use, yp_use

def _prep_eval_frame(df, feats, target):
    cols = [c for c in feats if c in df.columns]
    if not cols or target not in df.columns:
        return None, None

    work = df[cols + [target] + [c for c in ["CNPJ_CIA", "DT_REFER", "ANO"] if c in df.columns]].copy()
    work = work.replace([np.inf, -np.inf], np.nan)
    work[cols] = work[cols].apply(pd.to_numeric, errors="coerce")
    work[target] = pd.to_numeric(work[target], errors="coerce")
    work = work.dropna(subset=[target])

    if work.empty:
        return None, None

    X = work[cols].astype(float)
    y = work[target].astype(float)
    return work, (X, y)

# ---------------------------------------------------------------------
# Split de calibração / teste final
# ---------------------------------------------------------------------
if "ANO" in teste.columns and teste["ANO"].notna().any():
    anos = pd.to_numeric(teste["ANO"], errors="coerce").dropna().astype(int).unique().tolist()
    anos = sorted(anos)

    if len(anos) >= 2:
        ANO_CALIB = anos[-2]
        ANO_TEST2 = anos[-1]
        calib = teste[pd.to_numeric(teste["ANO"], errors="coerce") == ANO_CALIB].copy()
        test2 = teste[pd.to_numeric(teste["ANO"], errors="coerce") == ANO_TEST2].copy()
        print(f"Calibração ({ANO_CALIB}): {len(calib):,} obs | Teste final ({ANO_TEST2}): {len(test2):,} obs")
        _reg_evento("Etapa 8b", "split temporal definido", ano_calibracao=ANO_CALIB, ano_teste=ANO_TEST2)
    else:
        np.random.seed(42)
        idx_c = np.random.choice(len(teste), max(len(teste) // 2, 1), replace=False)
        mask = np.zeros(len(teste), dtype=bool)
        mask[idx_c] = True
        calib = teste[mask].copy()
        test2 = teste[~mask].copy()
        print("⚠️  Apenas um ano disponível — split 50/50 aleatório")
        _reg_evento("Etapa 8b", "split aleatório por falta de anos suficientes", nivel="warning")
else:
    np.random.seed(42)
    idx_c = np.random.choice(len(teste), max(len(teste) // 2, 1), replace=False)
    mask = np.zeros(len(teste), dtype=bool)
    mask[idx_c] = True
    calib = teste[mask].copy()
    test2 = teste[~mask].copy()
    print("⚠️  ANO ausente — split 50/50 aleatório")
    _reg_evento("Etapa 8b", "split aleatório por ausência de ANO", nivel="warning")

rows_cp = []
resumo_cp_target = []
falhas_cp = []

print(f"\nConformal: {len(TARGETS)} targets potenciais | calib={len(calib):,} | teste={len(test2):,}")
_reg_evento("Etapa 8b", "conformal iniciado", calib=int(len(calib)), teste=int(len(test2)), targets=int(len(TARGETS)))

for target in TARGETS:
    if target not in melhores:
        continue

    alg_nome = melhores[target]
    obj, modelo, err = _carregar_modelo_artifato(target, alg_nome)
    if err:
        falhas_cp.append({"target": target, "erro": err})
        logger.warning("Etapa 8b | %s | target=%s", err, target)
        continue

    schema_feats = _schema_do_modelo(obj, modelo)
    if not schema_feats:
        falhas_cp.append({"target": target, "erro": "schema do modelo não identificado"})
        logger.warning("Etapa 8b | schema do modelo não identificado | target=%s", target)
        continue

    schema_feats = _dedup_preservando_ordem([c for c in schema_feats if isinstance(c, str)])
    schema_hash = _hash_schema(schema_feats)

    perturb_feats = _perturbacao_do_artefato(obj, selected_features_por_target.get(target, []))
    perturb_feats = [f for f in perturb_feats if f in schema_feats]
    perturb_feats = _dedup_preservando_ordem(perturb_feats)

    mediana_treino, bounds_treino = _stats_treino(schema_feats)

    prep_c = _prep_eval_frame(calib, schema_feats, target)
    prep_t = _prep_eval_frame(test2, schema_feats, target)
    if prep_c[0] is None or prep_t[0] is None:
        falhas_cp.append({"target": target, "erro": "frame de calibração/teste vazio após preparação"})
        logger.warning("Etapa 8b | frame vazio após preparação | target=%s | schema=%d", target, len(schema_feats))
        continue

    df_c, (Xc, yc) = prep_c
    df_t2, (Xt2, yt2) = prep_t

    # completa valores faltantes com mediana do treino
    Xc = Xc.copy()
    Xt2 = Xt2.copy()
    for c in schema_feats:
        if c not in Xc.columns:
            Xc[c] = mediana_treino.get(c, np.nan)
        if c not in Xt2.columns:
            Xt2[c] = mediana_treino.get(c, np.nan)

    Xc = Xc[schema_feats].apply(pd.to_numeric, errors="coerce")
    Xt2 = Xt2[schema_feats].apply(pd.to_numeric, errors="coerce")

    for c in schema_feats:
        fillv = mediana_treino.get(c, 0.0)
        if pd.isna(fillv):
            fillv = 0.0
        Xc[c] = Xc[c].fillna(fillv)
        Xt2[c] = Xt2[c].fillna(fillv)

    # garante exatamente o mesmo schema esperado pelo pipeline
    Xc = Xc.reindex(columns=schema_feats, fill_value=0.0)
    Xt2 = Xt2.reindex(columns=schema_feats, fill_value=0.0)

    if Xc.shape[1] != len(schema_feats) or Xt2.shape[1] != len(schema_feats):
        falhas_cp.append({"target": target, "erro": "schema inconsistente após reindex"})
        logger.warning(
            "Etapa 8b | schema inconsistente após reindex | target=%s | shape_calib=%s | shape_teste=%s | schema=%d",
            target, Xc.shape, Xt2.shape, len(schema_feats)
        )
        continue

    # previsões brutas do pipeline
    try:
        yc_pred_raw = np.asarray(modelo.predict(Xc), dtype=float)
        yt2_pred_raw = np.asarray(modelo.predict(Xt2), dtype=float)
    except Exception as e:
        falhas_cp.append({"target": target, "erro": str(e)})
        logger.warning(
            "Etapa 8b | falha de predição em %s | X=%d features | erro=%s",
            target, Xc.shape[1], e
        )
        continue

    # escolhe escala com base no comportamento do conjunto de calibração
    transf = get_transform(target)
    modo_escala, yc_use, yc_pred_use = _avaliar_modo_escala(yc.values, yc_pred_raw, transf)
    _, yt2_use, yt2_pred_use = _avaliar_modo_escala(yt2.values, yt2_pred_raw, transf)

    if modo_escala is None or yc_use is None or yc_pred_use is None:
        falhas_cp.append({"target": target, "erro": "não foi possível alinhar escala na calibração"})
        logger.warning(
            "Etapa 8b | falha de alinhamento de escala | target=%s | schema=%d | hash=%s",
            target, len(schema_feats), schema_hash
        )
        continue

    # usa a mesma lógica de escala para calibração e teste
    # se teste não bater bem na mesma escala, ainda assim seguimos com a escala escolhida na calibração
    if modo_escala == "raw":
        yt2_use = np.asarray(yt2.values, dtype=float)
        yt2_pred_use = np.asarray(yt2_pred_raw, dtype=float)
    else:
        try:
            yt2_use = np.asarray(inv_transform(np.asarray(yt2.values, dtype=float), transf), dtype=float)
            yt2_pred_use = np.asarray(inv_transform(np.asarray(yt2_pred_raw, dtype=float), transf), dtype=float)
        except Exception as e:
            falhas_cp.append({"target": target, "erro": f"falha ao inverter escala do teste: {e}"})
            logger.warning("Etapa 8b | falha de inversão no teste | target=%s | erro=%s", target, e)
            continue

    # filtra valores válidos
    mask_c = np.isfinite(yc_use) & np.isfinite(yc_pred_use)
    mask_t = np.isfinite(yt2_use) & np.isfinite(yt2_pred_use)

    if mask_c.sum() < 2 or mask_t.sum() < 1:
        falhas_cp.append({"target": target, "erro": "amostra insuficiente após limpeza"})
        logger.warning(
            "Etapa 8b | amostra insuficiente | target=%s | calib=%d | teste=%d",
            target, int(mask_c.sum()), int(mask_t.sum())
        )
        continue

    yc_use = yc_use[mask_c]
    yc_pred_use = yc_pred_use[mask_c]
    yt2_use = yt2_use[mask_t]
    yt2_pred_use = yt2_pred_use[mask_t]

    # intervalo conformal split: quantil dos resíduos de calibração
    scores = np.abs(yc_use - yc_pred_use)
    scores = scores[np.isfinite(scores)]
    if scores.size < 2:
        falhas_cp.append({"target": target, "erro": "scores de calibração insuficientes"})
        logger.warning("Etapa 8b | scores insuficientes | target=%s", target)
        continue

    n = len(scores)
    level = min(np.ceil((n + 1) * (1 - ALPHA_CP)) / n, 1.0)
    q_hat = float(np.quantile(scores, level))

    lb = yt2_pred_use - q_hat
    ub = yt2_pred_use + q_hat
    coberto = (yt2_use >= lb) & (yt2_use <= ub)

    cobertura_real = float(coberto.mean())
    largura_media = float((ub - lb).mean())
    smape_calib = float(np.nanmedian(2.0 * np.abs(yc_use - yc_pred_use) / (np.abs(yc_use) + np.abs(yc_pred_use) + 1e-12)))
    smape_teste = float(np.nanmedian(2.0 * np.abs(yt2_use - yt2_pred_use) / (np.abs(yt2_use) + np.abs(yt2_pred_use) + 1e-12)))

    base = target.replace("TARGET_", "").rsplit("_ITR", 1)[0].rsplit("_DFP", 1)[0]
    hor = next((h for h in _HORIZONTES if target.endswith(h)), "N/A")

    for i in range(len(yt2_use)):
        row = df_t2.iloc[i]
        rows_cp.append({
            "CNPJ_CIA": row.get("CNPJ_CIA"),
            "DT_REFER": row.get("DT_REFER"),
            "ANO": row.get("ANO"),
            "Target": target,
            "Variavel": NOME_VARIAVEL.get(base, base),
            "Horizonte": hor.replace("_", ""),
            "Algoritmo": alg_nome,
            "SchemaHash": schema_hash,
            "SchemaLen": int(len(schema_feats)),
            "ModoEscala": modo_escala,
            "y_true": float(yt2_use[i]),
            "y_pred": float(yt2_pred_use[i]),
            "lb_90": float(lb[i]),
            "ub_90": float(ub[i]),
            "width_90": float(ub[i] - lb[i]),
            "coberto": bool(coberto[i]),
            "q_hat": float(q_hat),
            "alpha": float(ALPHA_CP),
        })

    resumo_cp_target.append({
        "Target": target,
        "Variavel": NOME_VARIAVEL.get(base, base),
        "Horizonte": hor.replace("_", ""),
        "Algoritmo": alg_nome,
        "SchemaLen": int(len(schema_feats)),
        "SchemaHash": schema_hash,
        "ModoEscala": modo_escala,
        "N_calib": int(len(yc_use)),
        "N_teste": int(len(yt2_use)),
        "q_hat": float(q_hat),
        "CoberturaReal": cobertura_real,
        "LarguraMedia": largura_media,
        "SMAPE_calib_med": smape_calib,
        "SMAPE_teste_med": smape_teste,
    })

    print(
        f"{NOME_VARIAVEL.get(base, base):<22} [{hor.replace('_','')}] {alg_nome:<18} "
        f"schema={len(schema_feats):>3} | modo={modo_escala:<12} | "
        f"q̂={q_hat/1e6:>7.1f}Bi | Cob={cobertura_real:.1%} | Larg={largura_media/1e6:>7.1f}Bi"
    )
    _reg_evento(
        "Etapa 8b",
        "intervalo conformal calculado",
        target=target,
        algoritmo=alg_nome,
        schema=int(len(schema_feats)),
        modo=modo_escala,
        q_hat=float(q_hat),
        cobertura=float(cobertura_real),
        largura=float(largura_media)
    )

if rows_cp:
    df_cp = pd.DataFrame(rows_cp)
    df_cp.to_parquet(PASTA_SAIDA / "conformal_intervals.parquet", index=False)
    df_cp.to_csv(PASTA_SAIDA / "conformal_intervals.csv", index=False)

    resumo_cp = pd.DataFrame(resumo_cp_target)
    resumo_cp.to_csv(PASTA_SAIDA / "conformal_summary.csv", index=False)

    print(f"\nOK conformal_intervals.csv/parquet salvo ({len(df_cp):,} linhas)")
    print("\n=== Resumo por target ===")
    print(resumo_cp.sort_values(["Variavel", "Horizonte"]).round(4).to_string(index=False))

    print("\n=== Cobertura média por variável ===")
    print(
        resumo_cp.groupby("Variavel")["CoberturaReal"]
        .agg(["mean", "median", "count"])
        .round(4)
        .to_string()
    )

    print("\n=== Intervalo médio por variável (largura) ===")
    print(
        resumo_cp.groupby("Variavel")["LarguraMedia"]
        .agg(["mean", "median"])
        .round(4)
        .to_string()
    )

    _reg_evento(
        "Etapa 8b",
        "conformal concluído",
        linhas=int(len(df_cp)),
        targets=int(df_cp["Target"].nunique()),
        cobertura_media=float(df_cp["coberto"].mean())
    )
else:
    df_cp = pd.DataFrame()
    resumo_cp = pd.DataFrame()
    print("AVISO: conformal prediction sem resultados.")
    _reg_evento("Etapa 8b", "conformal sem resultados", nivel="warning", falhas=int(len(falhas_cp)))

if falhas_cp:
    df_falhas_cp = pd.DataFrame(falhas_cp)
    df_falhas_cp.to_csv(PASTA_SAIDA / "conformal_falhas.csv", index=False)
    print(f"\nFalhas registradas: {len(df_falhas_cp)}")
    print(df_falhas_cp.head(20).to_string(index=False))
else:
    df_falhas_cp = pd.DataFrame()

print("\nOK Etapa 8b concluida")

2026-06-05 16:36:52 | INFO     | [Etapa 8b] split temporal definido | {'ano_calibracao': 2024, 'ano_teste': 2025}
2026-06-05 16:36:52 | INFO     | [Etapa 8b] conformal iniciado | {'calib': 98, 'teste': 95, 'targets': 36}


Calibração (2024): 98 obs | Teste final (2025): 95 obs

Conformal: 36 targets potenciais | calib=98 | teste=95
Receita Líquida        [ITRT1] GradientBoosting   schema=153 | modo=raw          | q̂=  100.1Bi | Cob=90.5% | Larg=  200.3Bi


2026-06-05 16:36:52 | INFO     | [Etapa 8b] intervalo conformal calculado | {'target': 'TARGET_DRE_3.01_ITR_T1', 'algoritmo': 'GradientBoosting', 'schema': 153, 'modo': 'raw', 'q_hat': 100142113.21589485, 'cobertura': 0.9052631578947369, 'largura': 200284226.43178964}
2026-06-05 16:36:52 | INFO     | [Etapa 8b] intervalo conformal calculado | {'target': 'TARGET_DRE_3.01_ITR_T2', 'algoritmo': 'GradientBoosting', 'schema': 154, 'modo': 'raw', 'q_hat': 114871731.37661582, 'cobertura': 0.8958333333333334, 'largura': 229743462.7532316}


Receita Líquida        [ITRT2] GradientBoosting   schema=154 | modo=raw          | q̂=  114.9Bi | Cob=89.6% | Larg=  229.7Bi


2026-06-05 16:36:52 | INFO     | [Etapa 8b] intervalo conformal calculado | {'target': 'TARGET_DRE_3.01_ITR_T3', 'algoritmo': 'GradientBoosting', 'schema': 155, 'modo': 'raw', 'q_hat': 138748981.32284072, 'cobertura': 0.9583333333333334, 'largura': 277497962.64568144}
2026-06-05 16:36:53 | INFO     | [Etapa 8b] intervalo conformal calculado | {'target': 'TARGET_DRE_3.01_DFP', 'algoritmo': 'GradientBoosting', 'schema': 155, 'modo': 'raw', 'q_hat': 217067768.71196836, 'cobertura': 0.9565217391304348, 'largura': 434135537.42393667}


...

FCO                    [ITRT1] GradientBoosting   schema=153 | modo=raw          | q̂=    9.7Bi | Cob=87.4% | Larg=   19.5Bi
FCO                    [ITRT2] GradientBoosting   schema=156 | modo=raw          | q̂=   10.3Bi | Cob=91.7% | Larg=   20.7Bi


2026-06-05 16:36:59 | INFO     | [Etapa 8b] intervalo conformal calculado | {'target': 'TARGET_DFC_MI_6.01_ITR_T3', 'algoritmo': 'GradientBoosting', 'schema': 156, 'modo': 'raw', 'q_hat': 12792504.45090638, 'cobertura': 0.9583333333333334, 'largura': 25585008.90181276}
2026-06-05 16:37:00 | INFO     | [Etapa 8b] intervalo conformal calculado | {'target': 'TARGET_DFC_MI_6.01_DFP', 'algoritmo': 'GradientBoosting', 'schema': 154, 'modo': 'raw', 'q_hat': 22951123.88912487, 'cobertura': 0.9130434782608695, 'largura': 45902247.77824974}


FCO                    [ITRT3] GradientBoosting   schema=156 | modo=raw          | q̂=   12.8Bi | Cob=95.8% | Larg=   25.6Bi
FCO                    [DFP] GradientBoosting   schema=154 | modo=raw          | q̂=   23.0Bi | Cob=91.3% | Larg=   45.9Bi


2026-06-05 16:37:00 | INFO     | [Etapa 8b] conformal concluído | {'linhas': 2124, 'targets': 36, 'cobertura_media': 0.911487758945386}



OK conformal_intervals.csv/parquet salvo (2,124 linhas)

=== Resumo por target ===
                   Target           Variavel Horizonte        Algoritmo  SchemaLen   SchemaHash ModoEscala  N_calib  N_teste            q_hat  CoberturaReal     LarguraMedia  SMAPE_calib_med  SMAPE_teste_med
      TARGET_BPA_1.01_DFP   Ativo Circulante       DFP GradientBoosting        156 9cd75faea1e4        raw       96       69  62,983,328.6961         0.9130 125,966,657.3923           2.0000           2.0000
   TARGET_BPA_1.01_ITR_T1   Ativo Circulante     ITRT1     RandomForest        152 dcf777113362        raw       98       95  67,330,822.7879         0.9158 134,661,645.5758           2.0000           2.0000
   TARGET_BPA_1.01_ITR_T2   Ativo Circulante     ITRT2     RandomForest        153 aa0bf9de8dfc        raw       97       48  67,338,018.3621         0.9167 134,676,036.7242           2.0000           2.0000
   TARGET_BPA_1.01_ITR_T3   Ativo Circulante     ITRT3 GradientBoosting        157 9

## Etapa 9. Feature Importance Agregada com Ranking por Família

In [11]:
df_feat_rank = pd.DataFrame()

if feature_importances:
    acum = {}
    for target, info in feature_importances.items():
        if not info or 'importancias' not in info:
            continue
        for feat, val in info['importancias'].items():
            if pd.notna(val):
                acum[feat] = acum.get(feat, 0.0) + float(val)

    df_feat_rank = (
        pd.Series(acum)
        .reset_index()
        .rename(columns={'index': 'feature', 0: 'importancia_total'})
        .sort_values('importancia_total', ascending=False)
        .reset_index(drop=True)
    )
    df_feat_rank['rank'] = df_feat_rank.index + 1

    def familia(f):
        if f.startswith('macro_'):
            return 'Macro'
        if '_yoy' in f:
            return 'YoY'
        if '_lag' in f or '_roll' in f:
            return 'Lag/Roll'
        if f.startswith('ratio_') or '_over_' in f:
            return 'Razão/Cruzada'
        if f.startswith('pos_setor_'):
            return 'Posição Setor'
        if f.startswith('setor_'):
            return 'Setor dummy'
        if f.startswith('tri_'):
            return 'Sazonalidade'
        if f in ['flag_dfp', 'flag_covid', 'ano_norm']:
            return 'Temporal'
        if f in (KPIS or []):
            return 'KPI base'
        return 'Outro'

    df_feat_rank['familia'] = df_feat_rank['feature'].apply(familia)
    df_feat_rank.to_csv(PASTA_SAIDA / 'feature_importance_ranking.csv', index=False)

    top25 = df_feat_rank.head(25)
    print('=== Top-25 Features por Importância Agregada (todos os 36 targets) ===')
    print(top25[['rank', 'feature', 'familia', 'importancia_total']].to_string(index=False))
    print('\nImportância total por família:')
    print(
        df_feat_rank.groupby('familia')['importancia_total']
        .agg(['sum', 'count', 'mean'])
        .sort_values('sum', ascending=False)
        .round(4)
        .to_string()
    )
    registrar_evento('Etapa 9', 'feature importance consolidada', features=int(len(df_feat_rank)), familias=int(df_feat_rank['familia'].nunique()))
else:
    print('⚠️  feature_importances.pkl não disponível.')
    registrar_evento('Etapa 9', 'feature_importances indisponível', nivel='warning')

print('\n✅ Etapa 9 concluída')


2026-06-05 16:37:14 | INFO     | [Etapa 9] feature importance consolidada | {'features': 261, 'familias': 10}


=== Top-25 Features por Importância Agregada (todos os 36 targets) ===
 rank                           feature  familia  importancia_total
    1          TARGET_BPA_1_ITR_T3_lag1 Lag/Roll             3.7225
    2       TARGET_BPA_1.01_ITR_T3_lag1 Lag/Roll             1.9242
    3       TARGET_BPP_2.03_ITR_T3_lag1 Lag/Roll             1.8485
    4          TARGET_BPP_2_ITR_T2_lag2 Lag/Roll             1.7600
    5          TARGET_BPP_2_ITR_T3_lag2 Lag/Roll             1.6750
    6       TARGET_DRE_3.01_ITR_T3_lag1 Lag/Roll             1.3512
    7       TARGET_BPA_1.01_ITR_T1_lag1 Lag/Roll             1.2762
    8       TARGET_BPP_2.01_ITR_T2_lag1 Lag/Roll             1.0455
    9       TARGET_BPA_1.01_ITR_T2_lag1 Lag/Roll             1.0230
   10       TARGET_BPP_2.01_ITR_T3_lag1 Lag/Roll             0.9652
   11       TARGET_BPP_2.03_ITR_T1_lag1 Lag/Roll             0.9553
   12       TARGET_BPP_2.03_ITR_T2_lag1 Lag/Roll             0.9176
   13       TARGET_BPP_2.01_ITR_T2_lag2 Lag/R

## Etapa 10. Análise de Resíduos e Teste de Viés Sistemático

In [12]:
if not df_pred.empty:
    df_res_best = df_pred[df_pred.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)].copy()

    if not df_res_best.empty:
        detalhes = []
        resumo_rows = []

        print('=== Estatísticas de Resíduos (melhor modelo por target) ===')

        for target, grp in df_res_best.groupby('Target'):
            ali = alinhar_series(target, grp['y_true'].values, grp['y_pred'].values, modo='auto')
            if ali['n'] < 2:
                continue

            # Aviso quando alinhar_series escolhe raw_raw (escalas mistas).
            # raw_raw significa y_true em log-space e y_pred em R$ — pode ocorrer
            # se inv_transform resultar em overflow. Registra para auditoria.
            if ali['modo'] == 'raw_raw':
                logger.warning(
                    'Etapa 10 | modo raw_raw (escalas mistas possiveis) | '
                    'target=%s | ratio=%.3f | score=%.2f',
                    target, ali.get('ratio', float('nan')), ali.get('score', float('nan'))
                )
            yt = ali['yt']
            yp = ali['yp']
            residuo = yt - yp
            denom = (np.abs(yt) + np.abs(yp)) / 2.0
            erro_sim = np.where(denom > 1e-9, np.abs(residuo) / denom, np.nan)

            base = target.replace('TARGET_', '').rsplit('_ITR', 1)[0].rsplit('_DFP', 1)[0]
            hor = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')

            resumo_rows.append({
                'Target': target,
                'Variavel': NOME_VARIAVEL.get(base, base),
                'Horizonte': hor,
                'Algoritmo': melhores.get(target, '?'),
                'Modo': ali['modo'],
                'N_obs': int(ali['n']),
                'SMAPE': float(np.nanmean(erro_sim)),
                'residuo_media': float(np.nanmean(residuo)),
                'residuo_mediana': float(np.nanmedian(residuo)),
                'superestimacao_pct': float(np.mean(residuo < 0)),
                'score_alinhamento': float(ali['score']),
                'ratio_mediano': float(ali['ratio']) if pd.notna(ali['ratio']) else np.nan,
            })

            tmp = grp.copy().reset_index(drop=True)
            tmp['y_true_alinhado'] = yt
            tmp['y_pred_alinhado'] = yp
            tmp['residuo'] = residuo
            tmp['erro_sim'] = erro_sim
            tmp['Modo'] = ali['modo']
            tmp['Variavel'] = NOME_VARIAVEL.get(base, base)
            tmp['Horizonte'] = hor
            tmp['score_alinhamento'] = ali['score']
            tmp['ratio_mediano'] = ali['ratio']
            tmp['y_true_orig'] = tmp['y_true']
            tmp['y_pred_orig'] = tmp['y_pred']
            detalhes.append(tmp)

        if not detalhes:
            print('AVISO: não houve predições elegíveis para análise de resíduos.')
            registrar_evento('Etapa 10', 'resíduos sem resultados', nivel='warning')
        else:
            df_res_best = pd.concat(detalhes, ignore_index=True)
            resumo = pd.DataFrame(resumo_rows).sort_values(['Variavel', 'Horizonte'])
            resumo.to_csv(PASTA_SAIDA / 'residuos_resumo.csv', index=False)

            print(resumo.round(4).to_string(index=False))

            print('\n=== Teste de viés sistemático (H0: média do resíduo = 0) ===')
            for target, grp in df_res_best.groupby('Target'):
                sub = grp['residuo'].dropna()
                if len(sub) < 5:
                    continue

                t_stat, p_val = sp_stats.ttest_1samp(sub, 0.0, nan_policy='omit')
                media = float(sub.mean())
                mediana = float(sub.median())
                taxa_super = float((sub < 0).mean())  # superestimação do modelo
                base = target.replace('TARGET_', '').rsplit('_ITR', 1)[0].rsplit('_DFP', 1)[0]
                hor = next((h for h in _HORIZONTES if target.endswith(h)), '')
                modo = grp['Modo'].iloc[0]
                flag = ' VIES' if p_val < 0.05 else ''
                print(
                    f'  {NOME_VARIAVEL.get(base, base):<22} [{hor.replace("_","")}] '
                    f'modo={modo:<18} media={media:,.2f}  mediana={mediana:,.2f}  '
                    f't={t_stat:+.3f}  p={p_val:.4f}  super={taxa_super:.1%}{flag}'
                )

            # Correcao: NOME_CIA nao existe em predicoes_teste_detalhadas.parquet.
            # Adiciona via mapa_nome antes do groupby para evitar KeyError.
            if 'NOME_CIA' not in df_res_best.columns:
                df_res_best['NOME_CIA'] = df_res_best['CNPJ_CIA'].map(mapa_nome).fillna(df_res_best['CNPJ_CIA'])
            piores = df_res_best.groupby(['CNPJ_CIA', 'NOME_CIA'])['erro_sim'].mean().nlargest(5)
            print('\nTop-5 empresas com maior SMAPE medio:')
            print(piores.round(4).to_string())

            df_res_best.to_parquet(PASTA_SAIDA / 'residuos_detalhados.parquet', index=False)
            print(f'\nOK residuos_detalhados.parquet salvo ({len(df_res_best):,} linhas)')
            registrar_evento('Etapa 10', 'resíduos salvos', linhas=int(len(df_res_best)), resumo=int(len(resumo)))
    else:
        print('AVISO: não houve predições elegíveis para análise de resíduos.')
        registrar_evento('Etapa 10', 'não houve predições elegíveis', nivel='warning')
else:
    print('AVISO: predicoes_teste_detalhadas.parquet nao disponivel.')
    registrar_evento('Etapa 10', 'predicoes_teste_detalhadas ausente', nivel='warning')

print('\nOK Etapa 10 concluida')


2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1.01_DFP | ratio=0.805 | score=169.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1.01_ITR_T1 | ratio=0.992 | score=197.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1.01_ITR_T2 | ratio=0.968 | score=149.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1.01_ITR_T3 | ratio=1.003 | score=124.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1_DFP | ratio=0.878 | score=169.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1_ITR_T1 | ratio=1.013 | score=197.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_BPA_1_ITR_T2 | ratio=0.941 | score=149.00


=== Estatísticas de Resíduos (melhor modelo por target) ===


2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_DRE_3.01_ITR_T2 | ratio=1.119 | score=149.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_DRE_3.01_ITR_T3 | ratio=1.066 | score=124.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_DRE_3.11_DFP | ratio=0.776 | score=169.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_DRE_3.11_ITR_T1 | ratio=0.599 | score=197.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_DRE_3.11_ITR_T2 | ratio=0.522 | score=149.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_DRE_3.11_ITR_T3 | ratio=0.390 | score=124.00
2026-06-05 16:37:23 | WARNING  | Etapa 10 | modo raw_raw (escalas mistas possiveis) | target=TARGET_EBITDA_DFP | ratio=0.930 | score=

                   Target           Variavel Horizonte        Algoritmo    Modo  N_obs  SMAPE   residuo_media  residuo_mediana  superestimacao_pct  score_alinhamento  ratio_mediano
      TARGET_BPA_1.01_DFP   Ativo Circulante      _DFP GradientBoosting raw_raw    165 0.1760  1,885,618.7644     115,897.8476              0.4606           169.0000         0.8055
   TARGET_BPA_1.01_ITR_T1   Ativo Circulante   _ITR_T1     RandomForest raw_raw    193 0.1214   -673,423.5741     112,769.6045              0.4611           197.0000         0.9919
   TARGET_BPA_1.01_ITR_T2   Ativo Circulante   _ITR_T2     RandomForest raw_raw    145 0.1123   -326,859.3953     -85,250.0835              0.5172           149.0000         0.9676
   TARGET_BPA_1.01_ITR_T3   Ativo Circulante   _ITR_T3 GradientBoosting raw_raw    120 0.0873     79,188.6448     -19,781.2726              0.5083           124.0000         1.0034
         TARGET_BPA_1_DFP        Ativo Total      _DFP GradientBoosting raw_raw    165 0.1830 2

## Etapa 11. Visualizações (8 figuras)

In [13]:
import matplotlib.patches as mpatches

figdir = PASTA_SAIDA / 'figuras'
figdir.mkdir(parents=True, exist_ok=True)

PALETA_ALG = {
    'Ridge': '#3498db',
    'SVR': '#9b59b6',
    'RandomForest': '#2ecc71',
    'GradientBoosting': '#e74c3c'
}
CORES_FAM = {
    'Macro': '#e74c3c',
    'YoY': '#2ecc71',
    'Lag/Roll': '#3498db',
    'KPI base': '#f39c12',
    'Razão/Cruzada': '#1abc9c',
    'Setor dummy': '#9b59b6',
    'Posição Setor': '#e67e22',
    'Sazonalidade': '#95a5a6',
    'Temporal': '#bdc3c7',
    'Outro': '#ecf0f1'
}

def _save_fig(fig, nome):
    caminho = figdir / nome
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return caminho

# Fig 1 — Heatmap SMAPE por Algoritmo × Horizonte
if not df_te.empty and 'Horizonte' in df_te.columns:
    try:
        _c = next((c for c in df_te.columns if 'SMAPE_teste' in c), None)
        if _c is None:
            print('⚠️ Fig 1: coluna SMAPE_teste não encontrada em df_te.')
        else:
            pv = df_te.groupby(['Algoritmo', 'Horizonte'])[_c].mean().unstack()
            pv = pv.dropna(how='all', axis=0).dropna(how='all', axis=1)

            if pv.empty:
                print('⚠️ Fig 1: matriz vazia após agregação.')
            else:
                fig, ax = plt.subplots(figsize=(9, 4))
                sns.heatmap(
                    pv, annot=True, fmt='.3f', cmap='RdYlGn_r',
                    linewidths=0.5, linecolor='white', ax=ax,
                    cbar_kws={'label': 'SMAPE macro'}
                )
                ax.set_title(
                    'SMAPE macro por Algoritmo × Horizonte (36 targets)',
                    fontsize=11, fontweight='bold'
                )
                plt.tight_layout()
                _save_fig(fig, 'heatmap_smape_36targets.png')
                print('✅ Fig 1')
    except Exception as e:
        print(f'⚠️ Fig 1: {e}')
else:
    print('⚠️ Fig 1: df_te vazio ou sem coluna Horizonte.')

# Fig 2 — Heatmap SMAPE por Variável × Algoritmo (DFP)
if not df_te.empty:
    try:
        _c = next((c for c in df_te.columns if 'SMAPE_teste' in c), None)
        if _c is None:
            print('⚠️ Fig 2: coluna SMAPE_teste não encontrada em df_te.')
        elif 'base_variavel' not in df_te.columns:
            print('⚠️ Fig 2: coluna base_variavel ausente em df_te.')
        else:
            df_dfp = df_te[df_te['Horizonte'] == '_DFP'] if 'Horizonte' in df_te.columns else df_te.copy()
            pv = df_dfp.groupby(['base_variavel', 'Algoritmo'])[_c].mean().unstack()
            pv = pv.dropna(how='all', axis=0).dropna(how='all', axis=1)

            if pv.empty:
                print('⚠️ Fig 2: matriz vazia após agregação.')
            else:
                pv.index = [NOME_VARIAVEL.get(i, i) for i in pv.index]
                fig, ax = plt.subplots(figsize=(11, 5))
                sns.heatmap(
                    pv, annot=True, fmt='.3f', cmap='RdYlGn_r',
                    linewidths=0.5, linecolor='white', ax=ax,
                    cbar_kws={'label': 'SMAPE macro (DFP)'}
                )
                ax.set_title(
                    'SMAPE por Variável × Algoritmo — Horizonte DFP',
                    fontsize=11, fontweight='bold'
                )
                plt.tight_layout()
                _save_fig(fig, 'heatmap_smape_variaveis_dfp.png')
                print('✅ Fig 2')
    except Exception as e:
        print(f'⚠️ Fig 2: {e}')
else:
    print('⚠️ Fig 2: df_te vazio.')

# Fig 3 — Distribuição Z'' por setor
if not df_zscore.empty and 'SETOR' in df_zscore.columns:
    try:
        df_zp = df_zscore[df_zscore['altman_z_pp'].notna()].copy()
        setores = sorted(df_zp['SETOR'].dropna().unique())

        if not setores:
            print('⚠️ Fig 3: nenhum setor com Z\'\' válido.')
        else:
            fig, axes = plt.subplots(1, len(setores), figsize=(4 * len(setores), 5), sharey=False)
            if len(setores) == 1:
                axes = [axes]

            for ax, s in zip(axes, setores):
                d = df_zp[df_zp['SETOR'] == s]['altman_z_pp'].dropna()
                if d.empty:
                    ax.set_visible(False)
                    continue
                ax.hist(d, bins=15, edgecolor='white', alpha=0.85)
                ax.axvline(ZONA_CINZA_INF, ls='--', lw=1.5)
                ax.axvline(ZONA_SEGURA, ls='--', lw=1.5)
                ax.set_title(s, fontsize=10, fontweight='bold')
                ax.set_xlabel("Z''")

            plt.suptitle("Distribuição Z'' por Setor", fontsize=12, fontweight='bold')
            plt.tight_layout()
            _save_fig(fig, 'dist_zscore_setor.png')
            print('✅ Fig 3')
    except Exception as e:
        print(f'⚠️ Fig 3: {e}')
else:
    print("⚠️ Fig 3: df_zscore vazio ou sem coluna SETOR.")

# Fig 4 — Score de Risco por Setor
if 'score_risco' in dataset.columns and 'SETOR' in dataset.columns:
    try:
        base = dataset[dataset['score_risco'].notna()].copy()
        if base.empty:
            print('⚠️ Fig 4: dataset sem score_risco válido.')
        else:
            ord_s = base.groupby('SETOR')['score_risco'].median().sort_values(ascending=False).index
            fig, ax = plt.subplots(figsize=(9, 5))
            sns.boxplot(
                data=base,
                x='SETOR', y='score_risco', order=ord_s,
                palette='RdYlGn_r', ax=ax
            )
            for lim, cor, lbl in [
                (20, '#2ecc71', 'Baixo'),
                (40, '#f39c12', 'Moderado'),
                (60, '#e74c3c', 'Elevado')
            ]:
                ax.axhline(lim, ls=':', color=cor, lw=1.5, alpha=0.7, label=f'{lbl} ({lim})')
            ax.set_title('Score de Risco Composto por Setor', fontsize=12, fontweight='bold')
            ax.legend(fontsize=8)
            plt.tight_layout()
            _save_fig(fig, 'score_risco_setor.png')
            print('✅ Fig 4')
    except Exception as e:
        print(f'⚠️ Fig 4: {e}')
else:
    print('⚠️ Fig 4: colunas score_risco e/ou SETOR ausentes.')

# Fig 5 — Feature Importance Top-25
if not df_feat_rank.empty:
    try:
        top = df_feat_rank.head(25).copy()
        if top.empty:
            print('⚠️ Fig 5: df_feat_rank vazio.')
        else:
            cores = top['familia'].map(CORES_FAM).fillna('#bdc3c7')
            fig, ax = plt.subplots(figsize=(12, 8))
            ax.barh(range(len(top)), top['importancia_total'].values[::-1], color=cores.values[::-1])
            ax.set_yticks(range(len(top)))
            ax.set_yticklabels(top['feature'].values[::-1], fontsize=8)
            ax.set_xlabel('Importância Agregada (36 targets)', fontsize=10)
            ax.set_title('Top-25 Features por Importância Agregada', fontsize=12, fontweight='bold')
            ax.grid(axis='x', alpha=0.3)
            patches = [
                mpatches.Patch(color=v, label=k)
                for k, v in CORES_FAM.items()
                if k in top['familia'].values
            ]
            if patches:
                ax.legend(handles=patches, fontsize=8, loc='lower right')
            plt.tight_layout()
            _save_fig(fig, 'feature_importance_top25.png')
            print('✅ Fig 5')
    except Exception as e:
        print(f'⚠️ Fig 5: {e}')
else:
    print('⚠️ Fig 5: df_feat_rank vazio.')

# Fig 6 — Evolução Z'' — todas as empresas; destaque nas representativas
if not df_zscore.empty and 'ANO' in df_zscore.columns and 'NOME_CIA' in df_zscore.columns:
    try:
        emps_all = (
            df_zscore[df_zscore['altman_z_pp'].notna()]['NOME_CIA']
            .dropna()
            .unique()
            .tolist()
        )

        if not emps_all:
            print("⚠️ Fig 6: nenhuma empresa com Z'' válido.")
        else:
            emps_dest = (
                [v['nome'] if isinstance(v, dict) else v for v in EMPRESAS_DESTAQUE.values()]
                if 'EMPRESAS_DESTAQUE' in globals() and EMPRESAS_DESTAQUE else emps_all[:5]
            )

            df_ze = df_zscore[
                df_zscore['NOME_CIA'].isin(emps_all) &
                df_zscore['altman_z_pp'].notna()
            ].copy()

            if df_ze.empty:
                print("⚠️ Fig 6: df_ze vazio.")
            else:
                fig, ax = plt.subplots(figsize=(13, 6))
                cs = plt.cm.tab20(np.linspace(0, 1, max(len(emps_all), 1)))

                for i, emp in enumerate(emps_all):
                    s = df_ze[df_ze['NOME_CIA'] == emp].sort_values('ANO')
                    if s.empty:
                        continue
                    destaque = emp in emps_dest
                    ax.plot(
                        s['ANO'], s['altman_z_pp'],
                        marker='o' if destaque else None,
                        label=emp if destaque else '_nolegend_',
                        color=cs[i % len(cs)],
                        linewidth=2.5 if destaque else 1.0,
                        markersize=5 if destaque else 0,
                        alpha=1.0 if destaque else 0.35,
                        zorder=5 if destaque else 2
                    )
                    if destaque:
                        ax.annotate(
                            emp[:14],
                            xy=(s['ANO'].iloc[-1], s['altman_z_pp'].iloc[-1]),
                            xytext=(4, 2), textcoords='offset points',
                            fontsize=6, color=cs[i % len(cs)],
                            fontweight='bold'
                        )

                ax.axhspan(-10, ZONA_CINZA_INF, alpha=0.06)
                ax.axhspan(ZONA_CINZA_INF, ZONA_SEGURA, alpha=0.06)
                ax.axhspan(ZONA_SEGURA, 30, alpha=0.04)
                ax.axhline(ZONA_CINZA_INF, ls='--', lw=1.2)
                ax.axhline(ZONA_SEGURA, ls='--', lw=1.2)
                ax.set_xlabel('Ano')
                ax.set_ylabel("Z''")
                ax.set_title(
                    "Evolução Z'' — Todas as Empresas (2015–2025)\nLinhas destacadas = empresas representativas por setor",
                    fontsize=11, fontweight='bold'
                )
                ax.legend(fontsize=7, ncol=3, loc='upper left')
                ax.grid(alpha=0.3)
                plt.tight_layout()
                _save_fig(fig, 'evolucao_zscore_todas.png')
                print(f'✅ Fig 6 ({len(emps_all)} empresas, {len(emps_dest)} destacadas)')
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'⚠️ Fig 6: {e}')
else:
    print("⚠️ Fig 6: df_zscore vazio ou colunas ANO/NOME_CIA ausentes.")

# Fig 7 — Predito × Observado (escala original)
if not df_pred.empty:
    try:
        df_fp = df_pred[
            df_pred['Target'].isin(TARGETS_FOCO) &
            df_pred.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)
        ].copy()

        tgts_plot = [t for t in TARGETS_FOCO if t in df_fp['Target'].values][:12]

        if not tgts_plot:
            print('⚠️ Fig 7: nenhum target foco com predição disponível.')
        else:
            ncols = 4
            nrows = int(np.ceil(len(tgts_plot) / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
            axes = np.array(axes).reshape(-1)

            for idx, target in enumerate(tgts_plot):
                ax = axes[idx]
                sub = df_fp[df_fp['Target'] == target].dropna(subset=['y_true', 'y_pred']).copy()
                if sub.empty:
                    ax.set_visible(False)
                    continue

                transf = get_transform(target)
                yt = inv_transform(sub['y_true'].values, transf)
                yp = sub['y_pred'].values.astype(float)

                mask = np.isfinite(yt) & np.isfinite(yp)
                if mask.sum() < 2:
                    ax.set_visible(False)
                    continue

                yt = yt[mask]
                yp = yp[mask]

                alg = melhores.get(target, '?')
                r2v = _r2(yt, yp)
                spv = _smape(yt, yp)

                ax.scatter(
                    yp, yt, alpha=0.5, s=18, edgecolors='none',
                    color=PALETA_ALG.get(alg, '#3498db')
                )
                lm = min(yt.min(), yp.min())
                lM = max(yt.max(), yp.max())
                ax.plot([lm, lM], [lm, lM], 'k--', lw=1)

                base = target.replace('TARGET_', '').rsplit('_ITR', 1)[0].rsplit('_DFP', 1)[0]
                hor = next((h.replace('_', '') for h in _HORIZONTES if target.endswith(h)), '')
                ax.set_title(
                    f'{NOME_VARIAVEL.get(base, base)} [{hor}]\n{alg}  R²={r2v:.3f}  SMAPE={spv:.1%}',
                    fontsize=8
                )
                # Correcao: formatter usa /1e6 (bilhoes), nao mil.
                ax.set_xlabel('Predito (R$ bilhoes)', fontsize=7)
                ax.set_ylabel('Observado (R$ bilhoes)', fontsize=7)
                ax.tick_params(labelsize=7)

                if np.nanmax(np.abs(yt)) > 1e6:
                    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}Bi'))
                    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}Bi'))

            for j in range(len(tgts_plot), len(axes)):
                axes[j].set_visible(False)

            plt.suptitle(
                'Predito × Observado — Targets Foco TCC\n(escala original R$ mil)',
                fontsize=12, fontweight='bold'
            )
            plt.tight_layout()
            _save_fig(fig, 'predito_vs_observado_foco.png')
            print('✅ Fig 7')
    except Exception as e:
        print(f'⚠️ Fig 7: {e}')
else:
    print('⚠️ Fig 7: df_pred vazio.')

# Fig 8 — Intervalos Monte Carlo P5–P95 — todas as empresas por variável
if not df_stress.empty:
    try:
        for base_str in ['DRE_3.01', 'DRE_3.11', 'EBITDA']:
            t_alvo = f'TARGET_{base_str}_DFP'
            df_sf = df_stress[df_stress['target'] == t_alvo].copy()

            if df_sf.empty:
                print(f'⚠️ Fig 8: sem dados para {t_alvo}.')
                continue

            df_sf = df_sf.sort_values(['setor', 'empresa']).reset_index(drop=True)
            n = len(df_sf)

            emps_dest_nomes = (
                [v['nome'] if isinstance(v, dict) else v for v in EMPRESAS_DESTAQUE.values()]
                if 'EMPRESAS_DESTAQUE' in globals() and EMPRESAS_DESTAQUE else []
            )

            fig, ax = plt.subplots(figsize=(max(12, 0.6 * n), 5))
            x = np.arange(n)

            cores_bar = [
                '#e74c3c' if row['empresa'] in emps_dest_nomes else '#3498db'
                for _, row in df_sf.iterrows()
            ]

            ax.bar(x, df_sf['p50'] / 1e6, color=cores_bar, alpha=0.7, label='Mediana MC')
            ax.errorbar(
                x, df_sf['p50'] / 1e6,
                yerr=[(df_sf['p50'] - df_sf['p5']) / 1e6, (df_sf['p95'] - df_sf['p50']) / 1e6],
                fmt='none', color='#2c3e50', capsize=3, linewidth=1.2,
                label='IC [P5, P95]'
            )
            ax.scatter(
                x, df_sf['y_base'] / 1e6,
                color='#f39c12', zorder=5, s=40, label='Base (sem perturbação)'
            )
            ax.set_xticks(x)
            ax.set_xticklabels(
                [e[:12] + '…' if len(e) > 12 else e for e in df_sf['empresa']],
                rotation=45, ha='right', fontsize=7
            )
            ax.set_ylabel('R$ bilhões', fontsize=9)
            ax.set_title(
                f'Intervalos MC P5–P95 — {NOME_VARIAVEL.get(base_str, base_str)} DFP 2026\n'
                'Barras vermelhas = empresas representativas por setor',
                fontsize=10, fontweight='bold'
            )
            ax.legend(fontsize=8)
            ax.grid(axis='y', alpha=0.3)

            setores_unicos = df_sf['setor'].dropna().unique()
            if len(setores_unicos) > 0:
                y_text = ax.get_ylim()[0]
                for s in setores_unicos:
                    idx_s = df_sf[df_sf['setor'] == s].index
                    if len(idx_s) == 0:
                        continue
                    mid = (idx_s[0] + idx_s[-1]) / 2
                    ax.text(
                        mid, y_text, s[:10],
                        ha='center', fontsize=7, color='#555',
                        fontstyle='italic', transform=ax.transData
                    )

            plt.tight_layout()
            fname = f'intervalos_mc_{base_str.replace(".", "_")}.png'
            _save_fig(fig, fname)
            print(f'✅ Fig 8 ({NOME_VARIAVEL.get(base_str, base_str)}): {fname}')
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'⚠️ Fig 8: {e}')
else:
    print('⚠️ Fig 8: df_stress vazio.')

print('\n✅ Etapa 11 concluída — figuras processadas com validação de existência de dados')

✅ Fig 1
✅ Fig 2
✅ Fig 3
✅ Fig 4
✅ Fig 5
✅ Fig 6 (25 empresas, 5 destacadas)
✅ Fig 7
✅ Fig 8 (Receita Líquida): intervalos_mc_DRE_3_01.png
✅ Fig 8 (Lucro Líquido): intervalos_mc_DRE_3_11.png
✅ Fig 8 (EBITDA): intervalos_mc_EBITDA.png

✅ Etapa 11 concluída — figuras processadas com validação de existência de dados


## Etapa 12. Persistência Completa de Artefatos para o Script 5

In [14]:
# 12.1 Melhores modelos confirmados
with open(PASTA_SAIDA / 'melhores_modelos_v4.pkl', 'wb') as f:
    pickle.dump(melhores, f)
registrar_evento('Etapa 12', 'melhores modelos persistidos', arquivo='melhores_modelos_v4.pkl', total=int(len(melhores)))

# 12.2 Z-Score por empresa (resumo para Script 5)
if 'df_zscore' in globals() and not df_zscore.empty and 'altman_z_pp' in df_zscore.columns:
    zpe = {}
    _ca = 'ANO' if 'ANO' in df_zscore.columns else None

    base_z = df_zscore[df_zscore['altman_z_pp'].notna()].copy()
    for cnpj, grp in base_z.groupby('CNPJ_CIA'):
        grp_s = grp.sort_values(_ca) if _ca else grp
        ul = grp_s.iloc[-1]
        zpe[cnpj] = {
            'z_medio': float(grp['altman_z_pp'].mean()),
            'z_ultimo': float(ul['altman_z_pp']),
            'zona_ultimo': str(ul.get('zona_altman', 'N/D')),
            'nome': str(ul.get('NOME_CIA', cnpj)),
            'setor': str(ul.get('SETOR', ''))
        }

    with open(PASTA_SAIDA / 'zscore_por_empresa.pkl', 'wb') as f:
        pickle.dump(zpe, f)

    print(f'✅ zscore_por_empresa.pkl: {len(zpe)} empresas')
    registrar_evento('Etapa 12', 'zscore_por_empresa persistido', arquivo='zscore_por_empresa.pkl', empresas=int(len(zpe)))
else:
    zpe = {}
    print('⚠️ zscore_por_empresa.pkl não gerado: df_zscore vazio ou sem altman_z_pp.')
    registrar_evento('Etapa 12', 'zscore_por_empresa não gerado', nivel='warning')

# 12.3 Score de risco do dataset
if 'score_risco' in dataset.columns:
    cols_base = [c for c in ['CNPJ_CIA', 'NOME_CIA', 'ANO', 'SETOR',
                             'score_risco', 'classe_risco', 'altman_z_pp', 'zona_altman']
                 if c in dataset.columns]
    if cols_base:
        dataset[cols_base].to_parquet(PASTA_SAIDA / 'score_risco_dataset.parquet', index=False)
        print('✅ score_risco_dataset.parquet salvo')
        registrar_evento('Etapa 12', 'score_risco_dataset persistido', arquivo='score_risco_dataset.parquet', colunas=int(len(cols_base)))
    else:
        print('⚠️ score_risco_dataset.parquet não salvo: colunas base ausentes.')
        registrar_evento('Etapa 12', 'score_risco_dataset não salvo: colunas base ausentes', nivel='warning')
else:
    print('⚠️ score_risco_dataset.parquet não salvo: score_risco ausente.')
    registrar_evento('Etapa 12', 'score_risco_dataset não salvo: score_risco ausente', nivel='warning')

# 12.4 Feature importance ranking
if not df_feat_rank.empty:
    with open(PASTA_SAIDA / 'feature_importance_ranking.pkl', 'wb') as f:
        pickle.dump(df_feat_rank, f)

# 12.5 Relatório JSON completo
relatorio = {
    'versao': 'V4_AvaliacaoAprofundada',
    'run_id': RUN_ID,
    'data_execucao': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'targets_totais': len(TODOS_TARGETS),
    'targets_treinados': len(TARGETS),
    'targets_foco_tcc': len(TARGETS_FOCO),
    'n_treino': int(len(treino)),
    'n_teste': int(len(teste)),
    'melhores_modelos': {t: str(a) for t, a in melhores.items()},
    'contagem_vitorias': dict(Counter(melhores.values())),
    'zscore_altman': {
        'formula': "Z''=6.56X1+3.26X2+6.72X3+1.05X4",
        'limiar_segura': ZONA_SEGURA,
        'limiar_cinza': ZONA_CINZA_INF,
        'obs_validas': int(df_zscore['altman_z_pp'].notna().sum()) if 'df_zscore' in globals() and not df_zscore.empty else 0,
        'zonas': df_zscore['zona_altman'].value_counts().to_dict() if 'df_zscore' in globals() and not df_zscore.empty and 'zona_altman' in df_zscore.columns else {},
    },
    'score_risco': {
        # Correcao: kpis_disp pode nao existir se Cell 16 for pulada.
        # Usa globals().get() para evitar NameError.
        'n_kpis': len(globals().get('kpis_disp', [])),
        'kpis': globals().get('kpis_disp', []),
        'classes': dataset['classe_risco'].value_counts().to_dict() if 'classe_risco' in dataset.columns else {},
    },
    'estresse_mc': {
        'n_sim': N_SIM_MC,
        'sigma': SIGMA_MC,
        'combinacoes': len(res_stress) if 'res_stress' in globals() else 0
    },
    'feature_importance': {
        'n_features': len(df_feat_rank) if not df_feat_rank.empty else 0,
        'top10': df_feat_rank.head(10)['feature'].tolist() if not df_feat_rank.empty else [],
        'familias': df_feat_rank.groupby('familia')['importancia_total'].sum().to_dict() if not df_feat_rank.empty else {},
    },
    'arquivos_gerados': [
        'altman_zscore.csv',
        'altman_zscore.parquet',
        'score_risco_dataset.parquet',
        'analise_estresse_mc.csv',
        'analise_estresse_mc_resumo.csv',
        'metricas_por_setor.csv',
        'metricas_por_setor_diagnostico.csv',
        'resultados_teste_enriquecido.csv',
        'diagnostico_overfitting.csv',
        'feature_importance_ranking.csv',
        'residuos_detalhados.parquet',
        'residuos_resumo.csv',
        'melhores_modelos_v4.pkl',
        'zscore_por_empresa.pkl',
        'empresas_destaque.pkl',
        'conformal_intervals.csv',
        'conformal_intervals.parquet',
        'conformal_summary.csv',
    ],
    'stage_logs': STAGE_LOGS,
}

registrar_evento('Etapa 12', 'pipeline concluído e artefatos persistidos', targets=int(len(TARGETS)), artefatos=int(len(relatorio['arquivos_gerados'])))

logsdir = PASTA_SAIDA / 'logs'
logsdir.mkdir(parents=True, exist_ok=True)

with open(logsdir / 'relatorio_avaliacao_v4.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

pd.DataFrame(STAGE_LOGS).to_csv(logsdir / 'stage_logs.csv', index=False)
with open(logsdir / 'stage_logs.jsonl', 'w', encoding='utf-8') as f:
    for r in STAGE_LOGS:
        f.write(json.dumps(r, ensure_ascii=False, default=str) + '\n')

# ── Resumo final ──────────────────────────────────────────────────────────────
print('\n' + '═' * 80)
print('RESUMO — 04_cvm_avaliacao.ipynb')
print('═' * 80)
print(f'  Targets treinados (Script 3) : {len(TARGETS)} / {len(TODOS_TARGETS)} possíveis')
print(f'  Targets foco TCC             : {len(TARGETS_FOCO)} (Receita, Lucro, EBITDA × 4 horizontes)')

if 'melhores' in globals() and melhores:
    cont = Counter(melhores.values())
    _dom = max(cont, key=cont.get)
    print(f'  Algoritmo dominante          : {_dom} ({cont[_dom]}× melhor de {len(TARGETS)})')
else:
    print('  Algoritmo dominante          : indisponível')

if 'df_zscore' in globals() and not df_zscore.empty and 'altman_z_pp' in df_zscore.columns:
    nv = df_zscore['altman_z_pp'].notna().sum()
    ni = (df_zscore['zona_altman'] == 'Insolvencia').sum() if 'zona_altman' in df_zscore.columns else 0
    if nv > 0:
        print(f'  Z-Score Altman               : {nv:,} obs | {ni:,} em zona de insolvência ({ni / nv * 100:.1f}%)')

if 'df_stress' in globals() and not df_stress.empty:
    print(f'  Estresse Monte Carlo         : {len(df_stress)} combinações ({N_SIM_MC} sim. cada)')

print('─' * 80)
print('Artefatos em outputs/:')
for a in relatorio['arquivos_gerados']:
    print(f'  • {a}')
print('  • logs/pipeline_avaliacao.log')
print('  • logs/stage_logs.csv')
print('  • logs/stage_logs.jsonl')
print('  • logs/relatorio_avaliacao_v4.json')

print('═' * 80)
print('✅ Script 4 concluído — pronto para 05_cvm_cenarios.ipynb')
logger.info('Script 4 concluído | targets=%d | artefatos persistidos', len(TARGETS))


2026-06-05 16:38:38 | INFO     | [Etapa 12] melhores modelos persistidos | {'arquivo': 'melhores_modelos_v4.pkl', 'total': 36}
2026-06-05 16:38:38 | INFO     | [Etapa 12] zscore_por_empresa persistido | {'arquivo': 'zscore_por_empresa.pkl', 'empresas': 25}
2026-06-05 16:38:38 | INFO     | [Etapa 12] score_risco_dataset persistido | {'arquivo': 'score_risco_dataset.parquet', 'colunas': 6}
2026-06-05 16:38:38 | INFO     | [Etapa 12] pipeline concluído e artefatos persistidos | {'targets': 36, 'artefatos': 18}
2026-06-05 16:38:38 | INFO     | Script 4 concluído | targets=36 | artefatos persistidos


✅ zscore_por_empresa.pkl: 25 empresas
✅ score_risco_dataset.parquet salvo

════════════════════════════════════════════════════════════════════════════════
RESUMO — 04_cvm_avaliacao.ipynb
════════════════════════════════════════════════════════════════════════════════
  Targets treinados (Script 3) : 36 / 36 possíveis
  Targets foco TCC             : 12 (Receita, Lucro, EBITDA × 4 horizontes)
  Algoritmo dominante          : GradientBoosting (29× melhor de 36)
  Z-Score Altman               : 1,031 obs | 2 em zona de insolvência (0.2%)
  Estresse Monte Carlo         : 225 combinações (500 sim. cada)
────────────────────────────────────────────────────────────────────────────────
Artefatos em outputs/:
  • altman_zscore.csv
  • altman_zscore.parquet
  • score_risco_dataset.parquet
  • analise_estresse_mc.csv
  • analise_estresse_mc_resumo.csv
  • metricas_por_setor.csv
  • metricas_por_setor_diagnostico.csv
  • resultados_teste_enriquecido.csv
  • diagnostico_overfitting.csv
  • feature